# **GENES MAPPING**

In [ ]:
#!/usr/bin/env python3
# For each gene in /content/Genes.txt, search PubMed for:
# (GENE[Title/Abstract] OR GENE[MeSH Terms]) AND Psoriasis
# and save results (with PubMed links) to CSV + Excel.

import os
import time
import csv
import sys
import requests
import pandas as pd

# -------------------- CONFIG --------------------
NCBI_EMAIL = "raysona07@gmail.com"  # your email for NCBI
API_KEY = os.environ.get("NCBI_API_KEY", "").strip()

# Path to gene list (one gene per line, no header)
GENE_FILE = "/content/Genes.txt"

OUT_CSV  = "DGEs_psoriasis_pubmed.csv"
OUT_XLSX = "DGEs_psoriasis_pubmed.xlsx"

TOP_N = 5                       # how many top PMIDs per gene
SLEEP_PUBMED = 0.35 if API_KEY else 0.6  # respect NCBI rate limits
EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

# -------------------- BASIC HELPERS --------------------
def load_genes(path):
    """Load gene symbols from a text file (one per line)."""
    if not os.path.exists(path):
        print(f"❌ Gene file not found: {path}")
        return []
    genes = []
    with open(path, "r") as f:
        for line in f:
            g = line.strip()
            if g and not g.startswith("#"):
                genes.append(g)
    print(f"Loaded {len(genes)} genes.")
    return genes


def eutils_request(path, params, retries=3):
    """Wrapper for NCBI E-utilities with retries."""
    params = params.copy()
    params["email"] = NCBI_EMAIL
    if API_KEY:
        params["api_key"] = API_KEY
    url = f"{EUTILS}/{path}"
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            if attempt == retries:
                print(f"❌ NCBI request failed after {retries} attempts: {e}")
                raise
            time.sleep(SLEEP_PUBMED * attempt)


def esearch(query, retmax=50):
    """PubMed ESearch: return a list of PMIDs for a query."""
    params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "retmax": retmax,
        "sort": "relevance",
    }
    r = eutils_request("esearch.fcgi", params).json()
    return r.get("esearchresult", {}).get("idlist", [])


def esummary(pmids):
    """PubMed ESummary: title, journal, year for PMIDs."""
    if not pmids:
        return {}
    params = {"db": "pubmed", "id": ",".join(pmids), "retmode": "json"}
    r = eutils_request("esummary.fcgi", params).json()
    res = r.get("result", {})
    out = {}
    for pmid in pmids:
        d = res.get(pmid, {})
        title = d.get("title", "")
        journal = d.get("fulljournalname", d.get("source", ""))
        pubdate = d.get("pubdate", "")
        year = ""
        for token in str(pubdate).split():
            if token.isdigit() and len(token) == 4:
                year = token
                break
        out[pmid] = {"title": title, "journal": journal, "year": year}
    return out


def pubmed_url(pmid):
    return f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else ""


# -------------------- PUBMED QUERY + SEARCH --------------------
def build_pubmed_query_psoriasis(gene):
    """
    Build PubMed query:
    (GENE[Title/Abstract] OR GENE[MeSH Terms]) AND
    (psoriasis[Title/Abstract] OR psoriasis[MeSH Terms])
    """
    safe_gene = gene.replace('"', "")
    gene_block = f'({safe_gene}[Title/Abstract] OR {safe_gene}[MeSH Terms])'
    ps_block = '(psoriasis[Title/Abstract] OR psoriasis[MeSH Terms])'
    return f"{gene_block} AND {ps_block}"


def pubmed_search_gene_psoriasis(gene, top_n=TOP_N):
    """PubMed search for Gene + Psoriasis."""
    rows = []
    q = build_pubmed_query_psoriasis(gene)
    pmids = esearch(q, retmax=max(50, top_n))
    time.sleep(SLEEP_PUBMED)

    if not pmids:
        rows.append({
            "Gene": gene,
            "Known": "NO",
            "Source": "PubMed",
            "PMID": "NA",
            "Title": "NA",
            "Journal": "NA",
            "Year": "NA",
            "PubMed_URL": "NA",
            "Query": q,
            "Note": "No PubMed hits for Gene+Psoriasis"
        })
        return rows

    summaries = esummary(pmids)
    time.sleep(SLEEP_PUBMED)

    for pmid in pmids[:top_n]:
        m = summaries.get(pmid, {})
        rows.append({
            "Gene": gene,
            "Known": "YES",
            "Source": "PubMed",
            "PMID": pmid,
            "Title": m.get("title", ""),
            "Journal": m.get("journal", ""),
            "Year": m.get("year", ""),
            "PubMed_URL": pubmed_url(pmid),
            "Query": q,
            "Note": ""
        })

    return rows


# -------------------- MAIN --------------------
def main():
    genes = load_genes(GENE_FILE)
    if not genes:
        sys.exit(1)

    out_rows = []

    for i, gene in enumerate(genes, 1):
        print(f"[{i}/{len(genes)}] Checking gene: {gene}")

        gene_rows = []

        # PubMed
        try:
            pm_rows = pubmed_search_gene_psoriasis(gene, TOP_N)
            gene_rows.extend(pm_rows)
        except Exception as e:
            gene_rows.append({
                "Gene": gene,
                "Known": "ERROR",
                "Source": "PubMed",
                "PMID": "NA",
                "Title": "NA",
                "Journal": "NA",
                "Year": "NA",
                "PubMed_URL": "NA",
                "Query": "ERROR",
                "Note": f"PubMed error: {e}"
            })

        # If no YES row, add a summary NO
        has_known = any(r.get("Known") == "YES" for r in gene_rows)
        if not has_known:
            gene_rows.append({
                "Gene": gene,
                "Known": "NO",
                "Source": "Summary",
                "PMID": "NA",
                "Title": "NA",
                "Journal": "NA",
                "Year": "NA",
                "PubMed_URL": "NA",
                "Query": "",
                "Note": "No Gene+Psoriasis papers found in PubMed"
            })

        out_rows.extend(gene_rows)

    # -------------------- EXPORT --------------------
    cols = [
        "Gene", "Known", "Source", "PMID", "Title", "Journal", "Year",
        "PubMed_URL", "Query", "Note"
    ]

    df = pd.DataFrame(out_rows, columns=cols)
    df.replace('"', "'", regex=True, inplace=True)  # avoid weird quote issues

    # Excel
    df.to_excel(OUT_XLSX, index=False)

    # CSV with strong quoting
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, quoting=csv.QUOTE_ALL)
        w.writeheader()
        for row in out_rows:
            safe_row = {
                k: (str(v).replace('"', "'") if isinstance(v, str) else v)
                for k, v in row.items()
            }
            w.writerow(safe_row)

    print("\n✅ Completed!")
    print(f"Saved CSV : {OUT_CSV}")
    print(f"Saved XLSX: {OUT_XLSX}")


if __name__ == "__main__":
    main()


Loaded 556 genes.
[1/556] Checking gene: CCNB1
[2/556] Checking gene: CDK1
[3/556] Checking gene: CYP1A1
[4/556] Checking gene: CYP1B1
[5/556] Checking gene: CYP2E1
[6/556] Checking gene: MMP9
[7/556] Checking gene: PIK3R1
[8/556] Checking gene: UGT1A9
[9/556] Checking gene: ABCA9
[10/556] Checking gene: ACACB
[11/556] Checking gene: ACKR1
[12/556] Checking gene: ACP3
[13/556] Checking gene: ACP5
[14/556] Checking gene: ACSBG1
[15/556] Checking gene: ACTA2
[16/556] Checking gene: ACVR1C
[17/556] Checking gene: ADAMDEC1
[18/556] Checking gene: ADAMTS1
[19/556] Checking gene: ADAMTS9
[20/556] Checking gene: ADAMTS15
[21/556] Checking gene: ADAP2
[22/556] Checking gene: ADH1B
[23/556] Checking gene: ADIPOQ
[24/556] Checking gene: ADM
[25/556] Checking gene: ADRB1
[26/556] Checking gene: ADRB2
[27/556] Checking gene: AFTPH-DT
[28/556] Checking gene: AGTR1
[29/556] Checking gene: AHNAK2
[30/556] Checking gene: AIM2
[31/556] Checking gene: AKAP12
[32/556] Checking gene: AKR1B10
[33/556] Chec

In [ ]:
#!/usr/bin/env python3
# For each pathway in /content/Pathways.txt, search PubMed for:
# (PATHWAY[Title/Abstract]) AND Psoriasis
# and save results (with PubMed links) to CSV + Excel.

import os
import time
import csv
import sys
import requests
import pandas as pd

# -------------------- CONFIG --------------------
NCBI_EMAIL = "raysona07@gmail.com"  # your email for NCBI
API_KEY = os.environ.get("NCBI_API_KEY", "").strip()

# Path to pathway list (one pathway per line, no header)
PATHWAY_FILE = "/content/556.txt"

OUT_CSV  = "Pathways_556.csv"
OUT_XLSX = "Pathways_p556.xlsx"

TOP_N = 5                       # how many top PMIDs per pathway
SLEEP_PUBMED = 0.35 if API_KEY else 0.6  # respect NCBI rate limits
EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

# -------------------- BASIC HELPERS --------------------
def load_pathways(path):
    """Load pathway names from a text file (one per line)."""
    if not os.path.exists(path):
        print(f"❌ Pathway file not found: {path}")
        return []
    terms = []
    with open(path, "r") as f:
        for line in f:
            t = line.strip()
            if t and not t.startswith("#"):
                terms.append(t)
    print(f"Loaded {len(terms)} pathways.")
    return terms


def eutils_request(path, params, retries=3):
    """Wrapper for NCBI E-utilities with retries."""
    params = params.copy()
    params["email"] = NCBI_EMAIL
    if API_KEY:
        params["api_key"] = API_KEY
    url = f"{EUTILS}/{path}"
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            if attempt == retries:
                print(f"❌ NCBI request failed after {retries} attempts: {e}")
                raise
            time.sleep(SLEEP_PUBMED * attempt)


def esearch(query, retmax=50):
    """PubMed ESearch: return a list of PMIDs for a query."""
    params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "retmax": retmax,
        "sort": "relevance",
    }
    r = eutils_request("esearch.fcgi", params).json()
    return r.get("esearchresult", {}).get("idlist", [])


def esummary(pmids):
    """PubMed ESummary: title, journal, year for PMIDs."""
    if not pmids:
        return {}
    params = {"db": "pubmed", "id": ",".join(pmids), "retmode": "json"}
    r = eutils_request("esummary.fcgi", params).json()
    res = r.get("result", {})
    out = {}
    for pmid in pmids:
        d = res.get(pmid, {})
        title = d.get("title", "")
        journal = d.get("fulljournalname", d.get("source", ""))
        pubdate = d.get("pubdate", "")
        year = ""
        for token in str(pubdate).split():
            if token.isdigit() and len(token) == 4:
                year = token
                break
        out[pmid] = {"title": title, "journal": journal, "year": year}
    return out


def pubmed_url(pmid):
    return f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else ""


# -------------------- PUBMED QUERY + SEARCH --------------------
def build_pubmed_query_psoriasis_pathway(pathway):
    """
    Build PubMed query:
    PATHWAY[Title/Abstract] AND
    (psoriasis[Title/Abstract] OR psoriasis[MeSH Terms])
    """
    safe_pathway = pathway.replace('"', "")
    # pathway names are usually phrases; restricting to Title/Abstract is fine
    pathway_block = f'"{safe_pathway}"[Title/Abstract]'
    ps_block = '(psoriasis[Title/Abstract] OR psoriasis[MeSH Terms])'
    return f"{pathway_block} AND {ps_block}"


def pubmed_search_pathway_psoriasis(pathway, top_n=TOP_N):
    """PubMed search for Pathway + Psoriasis."""
    rows = []
    q = build_pubmed_query_psoriasis_pathway(pathway)
    pmids = esearch(q, retmax=max(50, top_n))
    time.sleep(SLEEP_PUBMED)

    if not pmids:
        rows.append({
            "Pathway": pathway,
            "Known": "NO",
            "Source": "PubMed",
            "PMID": "NA",
            "Title": "NA",
            "Journal": "NA",
            "Year": "NA",
            "PubMed_URL": "NA",
            "Query": q,
            "Note": "No PubMed hits for Pathway+Psoriasis"
        })
        return rows

    summaries = esummary(pmids)
    time.sleep(SLEEP_PUBMED)

    for pmid in pmids[:top_n]:
        m = summaries.get(pmid, {})
        rows.append({
            "Pathway": pathway,
            "Known": "YES",
            "Source": "PubMed",
            "PMID": pmid,
            "Title": m.get("title", ""),
            "Journal": m.get("journal", ""),
            "Year": m.get("year", ""),
            "PubMed_URL": pubmed_url(pmid),
            "Query": q,
            "Note": ""
        })

    return rows


# -------------------- MAIN --------------------
def main():
    pathways = load_pathways(PATHWAY_FILE)
    if not pathways:
        sys.exit(1)

    out_rows = []

    for i, pw in enumerate(pathways, 1):
        print(f"[{i}/{len(pathways)}] Checking pathway: {pw}")

        pw_rows = []

        try:
            pm_rows = pubmed_search_pathway_psoriasis(pw, TOP_N)
            pw_rows.extend(pm_rows)
        except Exception as e:
            pw_rows.append({
                "Pathway": pw,
                "Known": "ERROR",
                "Source": "PubMed",
                "PMID": "NA",
                "Title": "NA",
                "Journal": "NA",
                "Year": "NA",
                "PubMed_URL": "NA",
                "Query": "ERROR",
                "Note": f"PubMed error: {e}"
            })

        # If no YES row, add a summary NO (saying no evidence for psoriasis link)
        has_known = any(r.get("Known") == "YES" for r in pw_rows)
        if not has_known:
            pw_rows.append({
                "Pathway": pw,
                "Known": "NO",
                "Source": "Summary",
                "PMID": "NA",
                "Title": "NA",
                "Journal": "NA",
                "Year": "NA",
                "PubMed_URL": "NA",
                "Query": "",
                "Note": "No Pathway+Psoriasis papers found in PubMed"
            })

        out_rows.extend(pw_rows)

    # -------------------- EXPORT --------------------
    cols = [
        "Pathway", "Known", "Source", "PMID", "Title", "Journal", "Year",
        "PubMed_URL", "Query", "Note"
    ]

    df = pd.DataFrame(out_rows, columns=cols)
    df.replace('"', "'", regex=True, inplace=True)  # avoid weird quote issues

    # Excel
    df.to_excel(OUT_XLSX, index=False)

    # CSV with strong quoting
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, quoting=csv.QUOTE_ALL)
        w.writeheader()
        for row in out_rows:
            safe_row = {
                k: (str(v).replace('"', "'") if isinstance(v, str) else v)
                for k, v in row.items()
            }
            w.writerow(safe_row)

    print("\n✅ Completed!")
    print(f"Saved CSV : {OUT_CSV}")
    print(f"Saved XLSX: {OUT_XLSX}")


if __name__ == "__main__":
    main()


Loaded 90 pathways.
[1/90] Checking pathway: Term
[2/90] Checking pathway: IL-17 signaling pathway
[3/90] Checking pathway: Toll-like receptor signaling pathway
[4/90] Checking pathway: NOD-like receptor signaling pathway
[5/90] Checking pathway: C-type lectin receptor signaling pathway
[6/90] Checking pathway: TNF signaling pathway
[7/90] Checking pathway: Osteoclast differentiation
[8/90] Checking pathway: Apoptosis
[9/90] Checking pathway: RIG-I-like receptor signaling pathway
[10/90] Checking pathway: Cytosolic DNA-sensing pathway
[11/90] Checking pathway: Reactome Pathways
[12/90] Checking pathway: Interferon alpha/beta signaling
[13/90] Checking pathway: NOTCH3 Intracellular Domain Regulates Transcription
[14/90] Checking pathway: Signaling by NOTCH3
[15/90] Checking pathway: RUNX3 regulates NOTCH signaling
[16/90] Checking pathway: Biological Processes
[17/90] Checking pathway: Interleukin-27-Mediated Signaling Pathway
[18/90] Checking pathway: Antiviral Innate Immune Response
[

In [ ]:
#!/usr/bin/env python3
# For each pathway in /content/Pathways.txt, search PubMed for:
# (PATHWAY[Title/Abstract]) AND Psoriasis
# and save results (with PubMed links) to CSV + Excel.

import os
import time
import csv
import sys
import requests
import pandas as pd

# -------------------- CONFIG --------------------
NCBI_EMAIL = "raysona07@gmail.com"  # your email for NCBI
API_KEY = os.environ.get("NCBI_API_KEY", "").strip()

# Path to pathway list (one pathway per line, no header)
PATHWAY_FILE = "/content/179_p.txt"

OUT_CSV  = "Pathways_179.csv"
OUT_XLSX = "Pathways_179.xlsx"

TOP_N = 5                       # how many top PMIDs per pathway
SLEEP_PUBMED = 0.35 if API_KEY else 0.6  # respect NCBI rate limits
EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

# -------------------- BASIC HELPERS --------------------
def load_pathways(path):
    """Load pathway names from a text file (one per line)."""
    if not os.path.exists(path):
        print(f"❌ Pathway file not found: {path}")
        return []
    terms = []
    with open(path, "r") as f:
        for line in f:
            t = line.strip()
            if t and not t.startswith("#"):
                terms.append(t)
    print(f"Loaded {len(terms)} pathways.")
    return terms


def eutils_request(path, params, retries=3):
    """Wrapper for NCBI E-utilities with retries."""
    params = params.copy()
    params["email"] = NCBI_EMAIL
    if API_KEY:
        params["api_key"] = API_KEY
    url = f"{EUTILS}/{path}"
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            if attempt == retries:
                print(f"❌ NCBI request failed after {retries} attempts: {e}")
                raise
            time.sleep(SLEEP_PUBMED * attempt)


def esearch(query, retmax=50):
    """PubMed ESearch: return a list of PMIDs for a query."""
    params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "retmax": retmax,
        "sort": "relevance",
    }
    r = eutils_request("esearch.fcgi", params).json()
    return r.get("esearchresult", {}).get("idlist", [])


def esummary(pmids):
    """PubMed ESummary: title, journal, year for PMIDs."""
    if not pmids:
        return {}
    params = {"db": "pubmed", "id": ",".join(pmids), "retmode": "json"}
    r = eutils_request("esummary.fcgi", params).json()
    res = r.get("result", {})
    out = {}
    for pmid in pmids:
        d = res.get(pmid, {})
        title = d.get("title", "")
        journal = d.get("fulljournalname", d.get("source", ""))
        pubdate = d.get("pubdate", "")
        year = ""
        for token in str(pubdate).split():
            if token.isdigit() and len(token) == 4:
                year = token
                break
        out[pmid] = {"title": title, "journal": journal, "year": year}
    return out


def pubmed_url(pmid):
    return f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else ""


# -------------------- PUBMED QUERY + SEARCH --------------------
def build_pubmed_query_psoriasis_pathway(pathway):
    """
    Build PubMed query:
    PATHWAY[Title/Abstract] AND
    (psoriasis[Title/Abstract] OR psoriasis[MeSH Terms])
    """
    safe_pathway = pathway.replace('"', "")
    # pathway names are usually phrases; restricting to Title/Abstract is fine
    pathway_block = f'"{safe_pathway}"[Title/Abstract]'
    ps_block = '(psoriasis[Title/Abstract] OR psoriasis[MeSH Terms])'
    return f"{pathway_block} AND {ps_block}"


def pubmed_search_pathway_psoriasis(pathway, top_n=TOP_N):
    """PubMed search for Pathway + Psoriasis."""
    rows = []
    q = build_pubmed_query_psoriasis_pathway(pathway)
    pmids = esearch(q, retmax=max(50, top_n))
    time.sleep(SLEEP_PUBMED)

    if not pmids:
        rows.append({
            "Pathway": pathway,
            "Known": "NO",
            "Source": "PubMed",
            "PMID": "NA",
            "Title": "NA",
            "Journal": "NA",
            "Year": "NA",
            "PubMed_URL": "NA",
            "Query": q,
            "Note": "No PubMed hits for Pathway+Psoriasis"
        })
        return rows

    summaries = esummary(pmids)
    time.sleep(SLEEP_PUBMED)

    for pmid in pmids[:top_n]:
        m = summaries.get(pmid, {})
        rows.append({
            "Pathway": pathway,
            "Known": "YES",
            "Source": "PubMed",
            "PMID": pmid,
            "Title": m.get("title", ""),
            "Journal": m.get("journal", ""),
            "Year": m.get("year", ""),
            "PubMed_URL": pubmed_url(pmid),
            "Query": q,
            "Note": ""
        })

    return rows


# -------------------- MAIN --------------------
def main():
    pathways = load_pathways(PATHWAY_FILE)
    if not pathways:
        sys.exit(1)

    out_rows = []

    for i, pw in enumerate(pathways, 1):
        print(f"[{i}/{len(pathways)}] Checking pathway: {pw}")

        pw_rows = []

        try:
            pm_rows = pubmed_search_pathway_psoriasis(pw, TOP_N)
            pw_rows.extend(pm_rows)
        except Exception as e:
            pw_rows.append({
                "Pathway": pw,
                "Known": "ERROR",
                "Source": "PubMed",
                "PMID": "NA",
                "Title": "NA",
                "Journal": "NA",
                "Year": "NA",
                "PubMed_URL": "NA",
                "Query": "ERROR",
                "Note": f"PubMed error: {e}"
            })

        # If no YES row, add a summary NO (saying no evidence for psoriasis link)
        has_known = any(r.get("Known") == "YES" for r in pw_rows)
        if not has_known:
            pw_rows.append({
                "Pathway": pw,
                "Known": "NO",
                "Source": "Summary",
                "PMID": "NA",
                "Title": "NA",
                "Journal": "NA",
                "Year": "NA",
                "PubMed_URL": "NA",
                "Query": "",
                "Note": "No Pathway+Psoriasis papers found in PubMed"
            })

        out_rows.extend(pw_rows)

    # -------------------- EXPORT --------------------
    cols = [
        "Pathway", "Known", "Source", "PMID", "Title", "Journal", "Year",
        "PubMed_URL", "Query", "Note"
    ]

    df = pd.DataFrame(out_rows, columns=cols)
    df.replace('"', "'", regex=True, inplace=True)  # avoid weird quote issues

    # Excel
    df.to_excel(OUT_XLSX, index=False)

    # CSV with strong quoting
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, quoting=csv.QUOTE_ALL)
        w.writeheader()
        for row in out_rows:
            safe_row = {
                k: (str(v).replace('"', "'") if isinstance(v, str) else v)
                for k, v in row.items()
            }
            w.writerow(safe_row)

    print("\n✅ Completed!")
    print(f"Saved CSV : {OUT_CSV}")
    print(f"Saved XLSX: {OUT_XLSX}")


if __name__ == "__main__":
    main()


Loaded 59 pathways.
[1/59] Checking pathway: KEGG Pathways
[2/59] Checking pathway: Insulin resistance
[3/59] Checking pathway: Arginine and proline metabolism
[4/59] Checking pathway: Ovarian steroidogenesis
[5/59] Checking pathway: Insulin signaling pathway
[6/59] Checking pathway: Steroid hormone biosynthesis
[7/59] Checking pathway: Biological Processes
[8/59] Checking pathway: Positive Regulation of Glycolytic Process
[9/59] Checking pathway: Chloride Transmembrane Transport
[10/59] Checking pathway: Androgen Metabolic Process
[11/59] Checking pathway: Steroid Hormone Biosynthetic Process
[12/59] Checking pathway: Positive Regulation of Morphogenesis of an Epithelium
[13/59] Checking pathway: cAMP-mediated Signaling
[14/59] Checking pathway: Positive Regulation of D-glucose Import
[15/59] Checking pathway: Hormone Biosynthetic Process
[16/59] Checking pathway: Inorganic Anion Transmembrane Transport
[17/59] Checking pathway: Epithelium Development
[18/59] Checking pathway: Positiv

# **29 + 14**

In [ ]:
#!/usr/bin/env python3
# For each pathway in /content/Pathways.txt, search PubMed for:
# (PATHWAY[Title/Abstract]) AND Psoriasis
# and save results (with PubMed links) to CSV + Excel.

import os
import time
import csv
import sys
import requests
import pandas as pd

# -------------------- CONFIG --------------------
NCBI_EMAIL = "raysona07@gmail.com"  # your email for NCBI
API_KEY = os.environ.get("NCBI_API_KEY", "").strip()

# Path to pathway list (one pathway per line, no header)
PATHWAY_FILE = "/content/2914.txt"

OUT_CSV  = "Pathways_2914.csv"
OUT_XLSX = "Pathways_2914.xlsx"

TOP_N = 5                       # how many top PMIDs per pathway
SLEEP_PUBMED = 0.35 if API_KEY else 0.6  # respect NCBI rate limits
EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

# -------------------- BASIC HELPERS --------------------
def load_pathways(path):
    """Load pathway names from a text file (one per line)."""
    if not os.path.exists(path):
        print(f"❌ Pathway file not found: {path}")
        return []
    terms = []
    with open(path, "r") as f:
        for line in f:
            t = line.strip()
            if t and not t.startswith("#"):
                terms.append(t)
    print(f"Loaded {len(terms)} pathways.")
    return terms


def eutils_request(path, params, retries=3):
    """Wrapper for NCBI E-utilities with retries."""
    params = params.copy()
    params["email"] = NCBI_EMAIL
    if API_KEY:
        params["api_key"] = API_KEY
    url = f"{EUTILS}/{path}"
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            if attempt == retries:
                print(f"❌ NCBI request failed after {retries} attempts: {e}")
                raise
            time.sleep(SLEEP_PUBMED * attempt)


def esearch(query, retmax=50):
    """PubMed ESearch: return a list of PMIDs for a query."""
    params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "retmax": retmax,
        "sort": "relevance",
    }
    r = eutils_request("esearch.fcgi", params).json()
    return r.get("esearchresult", {}).get("idlist", [])


def esummary(pmids):
    """PubMed ESummary: title, journal, year for PMIDs."""
    if not pmids:
        return {}
    params = {"db": "pubmed", "id": ",".join(pmids), "retmode": "json"}
    r = eutils_request("esummary.fcgi", params).json()
    res = r.get("result", {})
    out = {}
    for pmid in pmids:
        d = res.get(pmid, {})
        title = d.get("title", "")
        journal = d.get("fulljournalname", d.get("source", ""))
        pubdate = d.get("pubdate", "")
        year = ""
        for token in str(pubdate).split():
            if token.isdigit() and len(token) == 4:
                year = token
                break
        out[pmid] = {"title": title, "journal": journal, "year": year}
    return out


def pubmed_url(pmid):
    return f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else ""


# -------------------- PUBMED QUERY + SEARCH --------------------
def build_pubmed_query_psoriasis_pathway(pathway):
    """
    Build PubMed query:
    PATHWAY[Title/Abstract] AND
    (psoriasis[Title/Abstract] OR psoriasis[MeSH Terms])
    """
    safe_pathway = pathway.replace('"', "")
    # pathway names are usually phrases; restricting to Title/Abstract is fine
    pathway_block = f'"{safe_pathway}"[Title/Abstract]'
    ps_block = '(psoriasis[Title/Abstract] OR psoriasis[MeSH Terms])'
    return f"{pathway_block} AND {ps_block}"


def pubmed_search_pathway_psoriasis(pathway, top_n=TOP_N):
    """PubMed search for Pathway + Psoriasis."""
    rows = []
    q = build_pubmed_query_psoriasis_pathway(pathway)
    pmids = esearch(q, retmax=max(50, top_n))
    time.sleep(SLEEP_PUBMED)

    if not pmids:
        rows.append({
            "Pathway": pathway,
            "Known": "NO",
            "Source": "PubMed",
            "PMID": "NA",
            "Title": "NA",
            "Journal": "NA",
            "Year": "NA",
            "PubMed_URL": "NA",
            "Query": q,
            "Note": "No PubMed hits for Pathway+Psoriasis"
        })
        return rows

    summaries = esummary(pmids)
    time.sleep(SLEEP_PUBMED)

    for pmid in pmids[:top_n]:
        m = summaries.get(pmid, {})
        rows.append({
            "Pathway": pathway,
            "Known": "YES",
            "Source": "PubMed",
            "PMID": pmid,
            "Title": m.get("title", ""),
            "Journal": m.get("journal", ""),
            "Year": m.get("year", ""),
            "PubMed_URL": pubmed_url(pmid),
            "Query": q,
            "Note": ""
        })

    return rows


# -------------------- MAIN --------------------
def main():
    pathways = load_pathways(PATHWAY_FILE)
    if not pathways:
        sys.exit(1)

    out_rows = []

    for i, pw in enumerate(pathways, 1):
        print(f"[{i}/{len(pathways)}] Checking pathway: {pw}")

        pw_rows = []

        try:
            pm_rows = pubmed_search_pathway_psoriasis(pw, TOP_N)
            pw_rows.extend(pm_rows)
        except Exception as e:
            pw_rows.append({
                "Pathway": pw,
                "Known": "ERROR",
                "Source": "PubMed",
                "PMID": "NA",
                "Title": "NA",
                "Journal": "NA",
                "Year": "NA",
                "PubMed_URL": "NA",
                "Query": "ERROR",
                "Note": f"PubMed error: {e}"
            })

        # If no YES row, add a summary NO (saying no evidence for psoriasis link)
        has_known = any(r.get("Known") == "YES" for r in pw_rows)
        if not has_known:
            pw_rows.append({
                "Pathway": pw,
                "Known": "NO",
                "Source": "Summary",
                "PMID": "NA",
                "Title": "NA",
                "Journal": "NA",
                "Year": "NA",
                "PubMed_URL": "NA",
                "Query": "",
                "Note": "No Pathway+Psoriasis papers found in PubMed"
            })

        out_rows.extend(pw_rows)

    # -------------------- EXPORT --------------------
    cols = [
        "Pathway", "Known", "Source", "PMID", "Title", "Journal", "Year",
        "PubMed_URL", "Query", "Note"
    ]

    df = pd.DataFrame(out_rows, columns=cols)
    df.replace('"', "'", regex=True, inplace=True)  # avoid weird quote issues

    # Excel
    df.to_excel(OUT_XLSX, index=False)

    # CSV with strong quoting
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, quoting=csv.QUOTE_ALL)
        w.writeheader()
        for row in out_rows:
            safe_row = {
                k: (str(v).replace('"', "'") if isinstance(v, str) else v)
                for k, v in row.items()
            }
            w.writerow(safe_row)

    print("\n✅ Completed!")
    print(f"Saved CSV : {OUT_CSV}")
    print(f"Saved XLSX: {OUT_XLSX}")


if __name__ == "__main__":
    main()


Loaded 191 pathways.
[1/191] Checking pathway: KEGG Pathways
[2/191] Checking pathway: IL-17 signaling pathway
[3/191] Checking pathway: Lipid and atherosclerosis
[4/191] Checking pathway: AGE-RAGE signaling pathway in diabetic complications
[5/191] Checking pathway: Non-alcoholic fatty liver disease
[6/191] Checking pathway: NOD-like receptor signaling pathway
[7/191] Checking pathway: Cytosolic DNA-sensing pathway
[8/191] Checking pathway: Hematopoietic cell lineage
[9/191] Checking pathway: Toll-like receptor signaling pathway
[10/191] Checking pathway: NF-kappa B signaling pathway
[11/191] Checking pathway: Th17 cell differentiation
[12/191] Checking pathway: HIF-1 signaling pathway
[13/191] Checking pathway: Cytokine-cytokine receptor interaction
[14/191] Checking pathway: TNF signaling pathway
[15/191] Checking pathway: FoxO signaling pathway
[16/191] Checking pathway: Biological Processes
[17/191] Checking pathway: Positive Regulation of Macromolecule Biosynthetic Process
[18/19

# **29**

In [ ]:
#!/usr/bin/env python3
# For each pathway in /content/Pathways.txt, search PubMed for:
# (PATHWAY[Title/Abstract]) AND Psoriasis
# and save results (with PubMed links) to CSV + Excel.

import os
import time
import csv
import sys
import requests
import pandas as pd

# -------------------- CONFIG --------------------
NCBI_EMAIL = "raysona07@gmail.com"  # your email for NCBI
API_KEY = os.environ.get("NCBI_API_KEY", "").strip()

# Path to pathway list (one pathway per line, no header)
PATHWAY_FILE = "/content/29_p.txt"

OUT_CSV  = "Pathways_29.csv"
OUT_XLSX = "Pathways_29.xlsx"

TOP_N = 5                       # how many top PMIDs per pathway
SLEEP_PUBMED = 0.35 if API_KEY else 0.6  # respect NCBI rate limits
EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

# -------------------- BASIC HELPERS --------------------
def load_pathways(path):
    """Load pathway names from a text file (one per line)."""
    if not os.path.exists(path):
        print(f"❌ Pathway file not found: {path}")
        return []
    terms = []
    with open(path, "r") as f:
        for line in f:
            t = line.strip()
            if t and not t.startswith("#"):
                terms.append(t)
    print(f"Loaded {len(terms)} pathways.")
    return terms


def eutils_request(path, params, retries=3):
    """Wrapper for NCBI E-utilities with retries."""
    params = params.copy()
    params["email"] = NCBI_EMAIL
    if API_KEY:
        params["api_key"] = API_KEY
    url = f"{EUTILS}/{path}"
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            if attempt == retries:
                print(f"❌ NCBI request failed after {retries} attempts: {e}")
                raise
            time.sleep(SLEEP_PUBMED * attempt)


def esearch(query, retmax=50):
    """PubMed ESearch: return a list of PMIDs for a query."""
    params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "retmax": retmax,
        "sort": "relevance",
    }
    r = eutils_request("esearch.fcgi", params).json()
    return r.get("esearchresult", {}).get("idlist", [])


def esummary(pmids):
    """PubMed ESummary: title, journal, year for PMIDs."""
    if not pmids:
        return {}
    params = {"db": "pubmed", "id": ",".join(pmids), "retmode": "json"}
    r = eutils_request("esummary.fcgi", params).json()
    res = r.get("result", {})
    out = {}
    for pmid in pmids:
        d = res.get(pmid, {})
        title = d.get("title", "")
        journal = d.get("fulljournalname", d.get("source", ""))
        pubdate = d.get("pubdate", "")
        year = ""
        for token in str(pubdate).split():
            if token.isdigit() and len(token) == 4:
                year = token
                break
        out[pmid] = {"title": title, "journal": journal, "year": year}
    return out


def pubmed_url(pmid):
    return f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else ""


# -------------------- PUBMED QUERY + SEARCH --------------------
def build_pubmed_query_psoriasis_pathway(pathway):
    """
    Build PubMed query:
    PATHWAY[Title/Abstract] AND
    (psoriasis[Title/Abstract] OR psoriasis[MeSH Terms])
    """
    safe_pathway = pathway.replace('"', "")
    # pathway names are usually phrases; restricting to Title/Abstract is fine
    pathway_block = f'"{safe_pathway}"[Title/Abstract]'
    ps_block = '(psoriasis[Title/Abstract] OR psoriasis[MeSH Terms])'
    return f"{pathway_block} AND {ps_block}"


def pubmed_search_pathway_psoriasis(pathway, top_n=TOP_N):
    """PubMed search for Pathway + Psoriasis."""
    rows = []
    q = build_pubmed_query_psoriasis_pathway(pathway)
    pmids = esearch(q, retmax=max(50, top_n))
    time.sleep(SLEEP_PUBMED)

    if not pmids:
        rows.append({
            "Pathway": pathway,
            "Known": "NO",
            "Source": "PubMed",
            "PMID": "NA",
            "Title": "NA",
            "Journal": "NA",
            "Year": "NA",
            "PubMed_URL": "NA",
            "Query": q,
            "Note": "No PubMed hits for Pathway+Psoriasis"
        })
        return rows

    summaries = esummary(pmids)
    time.sleep(SLEEP_PUBMED)

    for pmid in pmids[:top_n]:
        m = summaries.get(pmid, {})
        rows.append({
            "Pathway": pathway,
            "Known": "YES",
            "Source": "PubMed",
            "PMID": pmid,
            "Title": m.get("title", ""),
            "Journal": m.get("journal", ""),
            "Year": m.get("year", ""),
            "PubMed_URL": pubmed_url(pmid),
            "Query": q,
            "Note": ""
        })

    return rows


# -------------------- MAIN --------------------
def main():
    pathways = load_pathways(PATHWAY_FILE)
    if not pathways:
        sys.exit(1)

    out_rows = []

    for i, pw in enumerate(pathways, 1):
        print(f"[{i}/{len(pathways)}] Checking pathway: {pw}")

        pw_rows = []

        try:
            pm_rows = pubmed_search_pathway_psoriasis(pw, TOP_N)
            pw_rows.extend(pm_rows)
        except Exception as e:
            pw_rows.append({
                "Pathway": pw,
                "Known": "ERROR",
                "Source": "PubMed",
                "PMID": "NA",
                "Title": "NA",
                "Journal": "NA",
                "Year": "NA",
                "PubMed_URL": "NA",
                "Query": "ERROR",
                "Note": f"PubMed error: {e}"
            })

        # If no YES row, add a summary NO (saying no evidence for psoriasis link)
        has_known = any(r.get("Known") == "YES" for r in pw_rows)
        if not has_known:
            pw_rows.append({
                "Pathway": pw,
                "Known": "NO",
                "Source": "Summary",
                "PMID": "NA",
                "Title": "NA",
                "Journal": "NA",
                "Year": "NA",
                "PubMed_URL": "NA",
                "Query": "",
                "Note": "No Pathway+Psoriasis papers found in PubMed"
            })

        out_rows.extend(pw_rows)

    # -------------------- EXPORT --------------------
    cols = [
        "Pathway", "Known", "Source", "PMID", "Title", "Journal", "Year",
        "PubMed_URL", "Query", "Note"
    ]

    df = pd.DataFrame(out_rows, columns=cols)
    df.replace('"', "'", regex=True, inplace=True)  # avoid weird quote issues

    # Excel
    df.to_excel(OUT_XLSX, index=False)

    # CSV with strong quoting
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, quoting=csv.QUOTE_ALL)
        w.writeheader()
        for row in out_rows:
            safe_row = {
                k: (str(v).replace('"', "'") if isinstance(v, str) else v)
                for k, v in row.items()
            }
            w.writerow(safe_row)

    print("\n✅ Completed!")
    print(f"Saved CSV : {OUT_CSV}")
    print(f"Saved XLSX: {OUT_XLSX}")


if __name__ == "__main__":
    main()


Loaded 35 pathways.
[1/35] Checking pathway: KEGG
[2/35] Checking pathway: Asthma
[3/35] Checking pathway: Circadian rhythm
[4/35] Checking pathway: Allograft rejection
[5/35] Checking pathway: Biological Processes
[6/35] Checking pathway: "Positive Regulation of Wnt Signaling Pathway, Planar Cell Polarity Pathway (GO:2000096)"
[7/35] Checking pathway: Engulfment of Apoptotic Cell (GO:0043652)
[8/35] Checking pathway: Positive Regulation of Lymphocyte Apoptotic Process (GO:0070230)
[9/35] Checking pathway: "Regulation of Wnt Signaling Pathway, Planar Cell Polarity Pathway (GO:2000095)"
[10/35] Checking pathway: Positive Regulation of Non-Canonical Wnt Signaling Pathway (GO:2000052)
[11/35] Checking pathway: Positive Regulation of MHC Class II Biosynthetic Process (GO:0045348)
[12/35] Checking pathway: Peptide Antigen Assembly With MHC Class II Protein Complex (GO:0002503)
[13/35] Checking pathway: MHC Class II Protein Complex Assembly (GO:0002399)
[14/35] Checking pathway: Regulation o

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# **14**

In [ ]:
#!/usr/bin/env python3
# For each pathway in /content/Pathways.txt, search PubMed for:
# (PATHWAY[Title/Abstract]) AND Psoriasis
# and save results (with PubMed links) to CSV + Excel.

import os
import time
import csv
import sys
import requests
import pandas as pd

# -------------------- CONFIG --------------------
NCBI_EMAIL = "raysona07@gmail.com"  # your email for NCBI
API_KEY = os.environ.get("NCBI_API_KEY", "").strip()

# Path to pathway list (one pathway per line, no header)
PATHWAY_FILE = "/content/14_p.txt"

OUT_CSV  = "Pathways_14.csv"
OUT_XLSX = "Pathways_14.xlsx"

TOP_N = 5                       # how many top PMIDs per pathway
SLEEP_PUBMED = 0.35 if API_KEY else 0.6  # respect NCBI rate limits
EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

# -------------------- BASIC HELPERS --------------------
def load_pathways(path):
    """Load pathway names from a text file (one per line)."""
    if not os.path.exists(path):
        print(f"❌ Pathway file not found: {path}")
        return []
    terms = []
    with open(path, "r") as f:
        for line in f:
            t = line.strip()
            if t and not t.startswith("#"):
                terms.append(t)
    print(f"Loaded {len(terms)} pathways.")
    return terms


def eutils_request(path, params, retries=3):
    """Wrapper for NCBI E-utilities with retries."""
    params = params.copy()
    params["email"] = NCBI_EMAIL
    if API_KEY:
        params["api_key"] = API_KEY
    url = f"{EUTILS}/{path}"
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            if attempt == retries:
                print(f"❌ NCBI request failed after {retries} attempts: {e}")
                raise
            time.sleep(SLEEP_PUBMED * attempt)


def esearch(query, retmax=50):
    """PubMed ESearch: return a list of PMIDs for a query."""
    params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "retmax": retmax,
        "sort": "relevance",
    }
    r = eutils_request("esearch.fcgi", params).json()
    return r.get("esearchresult", {}).get("idlist", [])


def esummary(pmids):
    """PubMed ESummary: title, journal, year for PMIDs."""
    if not pmids:
        return {}
    params = {"db": "pubmed", "id": ",".join(pmids), "retmode": "json"}
    r = eutils_request("esummary.fcgi", params).json()
    res = r.get("result", {})
    out = {}
    for pmid in pmids:
        d = res.get(pmid, {})
        title = d.get("title", "")
        journal = d.get("fulljournalname", d.get("source", ""))
        pubdate = d.get("pubdate", "")
        year = ""
        for token in str(pubdate).split():
            if token.isdigit() and len(token) == 4:
                year = token
                break
        out[pmid] = {"title": title, "journal": journal, "year": year}
    return out


def pubmed_url(pmid):
    return f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else ""


# -------------------- PUBMED QUERY + SEARCH --------------------
def build_pubmed_query_psoriasis_pathway(pathway):
    """
    Build PubMed query:
    PATHWAY[Title/Abstract] AND
    (psoriasis[Title/Abstract] OR psoriasis[MeSH Terms])
    """
    safe_pathway = pathway.replace('"', "")
    # pathway names are usually phrases; restricting to Title/Abstract is fine
    pathway_block = f'"{safe_pathway}"[Title/Abstract]'
    ps_block = '(psoriasis[Title/Abstract] OR psoriasis[MeSH Terms])'
    return f"{pathway_block} AND {ps_block}"


def pubmed_search_pathway_psoriasis(pathway, top_n=TOP_N):
    """PubMed search for Pathway + Psoriasis."""
    rows = []
    q = build_pubmed_query_psoriasis_pathway(pathway)
    pmids = esearch(q, retmax=max(50, top_n))
    time.sleep(SLEEP_PUBMED)

    if not pmids:
        rows.append({
            "Pathway": pathway,
            "Known": "NO",
            "Source": "PubMed",
            "PMID": "NA",
            "Title": "NA",
            "Journal": "NA",
            "Year": "NA",
            "PubMed_URL": "NA",
            "Query": q,
            "Note": "No PubMed hits for Pathway+Psoriasis"
        })
        return rows

    summaries = esummary(pmids)
    time.sleep(SLEEP_PUBMED)

    for pmid in pmids[:top_n]:
        m = summaries.get(pmid, {})
        rows.append({
            "Pathway": pathway,
            "Known": "YES",
            "Source": "PubMed",
            "PMID": pmid,
            "Title": m.get("title", ""),
            "Journal": m.get("journal", ""),
            "Year": m.get("year", ""),
            "PubMed_URL": pubmed_url(pmid),
            "Query": q,
            "Note": ""
        })

    return rows


# -------------------- MAIN --------------------
def main():
    pathways = load_pathways(PATHWAY_FILE)
    if not pathways:
        sys.exit(1)

    out_rows = []

    for i, pw in enumerate(pathways, 1):
        print(f"[{i}/{len(pathways)}] Checking pathway: {pw}")

        pw_rows = []

        try:
            pm_rows = pubmed_search_pathway_psoriasis(pw, TOP_N)
            pw_rows.extend(pm_rows)
        except Exception as e:
            pw_rows.append({
                "Pathway": pw,
                "Known": "ERROR",
                "Source": "PubMed",
                "PMID": "NA",
                "Title": "NA",
                "Journal": "NA",
                "Year": "NA",
                "PubMed_URL": "NA",
                "Query": "ERROR",
                "Note": f"PubMed error: {e}"
            })

        # If no YES row, add a summary NO (saying no evidence for psoriasis link)
        has_known = any(r.get("Known") == "YES" for r in pw_rows)
        if not has_known:
            pw_rows.append({
                "Pathway": pw,
                "Known": "NO",
                "Source": "Summary",
                "PMID": "NA",
                "Title": "NA",
                "Journal": "NA",
                "Year": "NA",
                "PubMed_URL": "NA",
                "Query": "",
                "Note": "No Pathway+Psoriasis papers found in PubMed"
            })

        out_rows.extend(pw_rows)

    # -------------------- EXPORT --------------------
    cols = [
        "Pathway", "Known", "Source", "PMID", "Title", "Journal", "Year",
        "PubMed_URL", "Query", "Note"
    ]

    df = pd.DataFrame(out_rows, columns=cols)
    df.replace('"', "'", regex=True, inplace=True)  # avoid weird quote issues

    # Excel
    df.to_excel(OUT_XLSX, index=False)

    # CSV with strong quoting
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, quoting=csv.QUOTE_ALL)
        w.writeheader()
        for row in out_rows:
            safe_row = {
                k: (str(v).replace('"', "'") if isinstance(v, str) else v)
                for k, v in row.items()
            }
            w.writerow(safe_row)

    print("\n✅ Completed!")
    print(f"Saved CSV : {OUT_CSV}")
    print(f"Saved XLSX: {OUT_XLSX}")


if __name__ == "__main__":
    main()


Loaded 243 pathways.
[1/243] Checking pathway: KEGG
[2/243] Checking pathway: IL-17 signaling pathway
[3/243] Checking pathway: Rheumatoid arthritis
[4/243] Checking pathway: NOD-like receptor signaling pathway
[5/243] Checking pathway: Cytokine-cytokine receptor interaction
[6/243] Checking pathway: Longevity regulating pathway
[7/243] Checking pathway: Toll-like receptor signaling pathway
[8/243] Checking pathway: NF-kappa B signaling pathway
[9/243] Checking pathway: HIF-1 signaling pathway
[10/243] Checking pathway: TNF signaling pathway
[11/243] Checking pathway: FoxO signaling pathway
[12/243] Checking pathway: Cellular senescence
[13/243] Checking pathway: PI3K-Akt signaling pathway
[14/243] Checking pathway: Graft-versus-host disease
[15/243] Checking pathway: Intestinal immune network for IgA production
[16/243] Checking pathway: VEGF signaling pathway
[17/243] Checking pathway: Cytosolic DNA-sensing pathway
[18/243] Checking pathway: Inflammatory bowel disease
[19/243] Checki

In [ ]:
#!/usr/bin/env python3
import os, time, csv, sys, requests
import pandas as pd

NCBI_EMAIL = "raysona07@gmail.com"
API_KEY = os.environ.get("NCBI_API_KEY", "").strip()
GENE_FILE = "/content/Genes.txt"

OUT_CSV  = "genes_psoriasis_pubmed.csv"
OUT_XLSX = "genes_psoriasis_pubmed.xlsx"

TOP_N = 5
SLEEP_PUBMED = 0.2 if API_KEY else 0.4
EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

def load_genes(path):
    if not os.path.exists(path):
        print(f"❌ Gene file not found: {path}")
        return []
    genes = []
    with open(path) as f:
        for line in f:
            g = line.strip()
            if g and not g.startswith("#"):
                genes.append(g)
    print(f"Loaded {len(genes)} genes.")
    return genes

def eutils_request(path, params, retries=2):
    params = params.copy()
    params["email"] = NCBI_EMAIL
    if API_KEY:
        params["api_key"] = API_KEY
    url = f"{EUTILS}/{path}"
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=10)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            if attempt == retries:
                print(f"❌ NCBI request failed after {retries} attempts: {e}")
                raise
            time.sleep(SLEEP_PUBMED * attempt)

def esearch(query, retmax=50):
    params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "retmax": retmax,
        "sort": "relevance",
    }
    r = eutils_request("esearch.fcgi", params).json()
    return r.get("esearchresult", {}).get("idlist", [])

def esummary(pmids):
    if not pmids:
        return {}
    params = {"db": "pubmed", "id": ",".join(pmids), "retmode": "json"}
    r = eutils_request("esummary.fcgi", params).json()
    res = r.get("result", {})
    out = {}
    for pmid in pmids:
        d = res.get(pmid, {})
        pubdate = str(d.get("pubdate", ""))
        year = ""
        for token in pubdate.split():
            if token.isdigit() and len(token) == 4:
                year = token
                break
        out[pmid] = {
            "title": d.get("title", ""),
            "journal": d.get("fulljournalname", d.get("source", "")),
            "year": year
        }
    return out

def pubmed_url(pmid):
    return f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else ""

def build_pubmed_query_psoriasis(gene):
    safe_gene = gene.replace('"', "")
    gene_block = f'({safe_gene}[Title/Abstract] OR {safe_gene}[MeSH Terms])'
    ps_block = '(psoriasis[Title/Abstract] OR psoriasis[MeSH Terms])'
    return f"{gene_block} AND {ps_block}"

def pubmed_search_gene_psoriasis(gene, top_n=TOP_N):
    rows = []
    q = build_pubmed_query_psoriasis(gene)
    pmids = esearch(q, retmax=max(50, top_n))
    time.sleep(SLEEP_PUBMED)

    if not pmids:
        rows.append({
            "Gene": gene,
            "Known": "NO",
            "Source": "PubMed",
            "PMID": "NA",
            "Title": "NA",
            "Journal": "NA",
            "Year": "NA",
            "PubMed_URL": "NA",
            "Query": q,
            "Note": "No PubMed hits for Gene+Psoriasis"
        })
        return rows

    summaries = esummary(pmids)
    time.sleep(SLEEP_PUBMED)

    for pmid in pmids[:top_n]:
        m = summaries.get(pmid, {})
        rows.append({
            "Gene": gene,
            "Known": "YES",
            "Source": "PubMed",
            "PMID": pmid,
            "Title": m.get("title", ""),
            "Journal": m.get("journal", ""),
            "Year": m.get("year", ""),
            "PubMed_URL": pubmed_url(pmid),
            "Query": q,
            "Note": ""
        })
    return rows

def main():
    genes = load_genes(GENE_FILE)
    if not genes:
        sys.exit(1)

    out_rows = []
    for i, gene in enumerate(genes, 1):
        print(f"[{i}/{len(genes)}] Checking gene: {gene}")
        try:
            gene_rows = pubmed_search_gene_psoriasis(gene, TOP_N)
        except Exception as e:
            gene_rows = [{
                "Gene": gene,
                "Known": "ERROR",
                "Source": "PubMed",
                "PMID": "NA",
                "Title": "NA",
                "Journal": "NA",
                "Year": "NA",
                "PubMed_URL": "NA",
                "Query": "ERROR",
                "Note": f"PubMed error: {e}"
            }]
        out_rows.extend(gene_rows)

    cols = ["Gene", "Known", "Source", "PMID", "Title", "Journal", "Year",
            "PubMed_URL", "Query", "Note"]
    df = pd.DataFrame(out_rows, columns=cols)
    df.to_excel(OUT_XLSX, index=False)
    df.to_csv(OUT_CSV, index=False, quoting=csv.QUOTE_ALL)

    print("\n✅ Completed!")
    print(f"Saved CSV : {OUT_CSV}")
    print(f"Saved XLSX: {OUT_XLSX}")

if __name__ == "__main__":
    main()


In [ ]:
# ---- Install required packages (only runs if missing) ----
packages <- c("rentrez", "readr", "dplyr")

new_pkgs <- packages[!(packages %in% installed.packages()[, "Package"])]
if (length(new_pkgs)) {
  install.packages(new_pkgs, repos = "https://cloud.r-project.org")
}

In [ ]:
# ---- Load libraries ----
library(rentrez)
library(readr)
library(dplyr)

# ---------- CONFIG ----------
email <- "your_email@example.com"     # 👈 Replace with your email (required by NCBI)
# Optional: Set NCBI API key to increase rate limits (get one from your NCBI account)
# Sys.setenv(NCBI_API_KEY = "your_ncbi_api_key_here")
Sys.setenv(NCBI_API_KEY = Sys.getenv("NCBI_API_KEY"))

condition <- "Psoriasis"              # 👈 Change to any condition you want
genes_csv <- "/content/genes.csv"              # CSV with column "Gene"
out_csv <- "pubmed_gene_condition_hits.csv"
query_template <- '("%s"[All Fields]) AND ("%s"[All Fields])'
sleep_sec <- if (nzchar(Sys.getenv("NCBI_API_KEY"))) 0.12 else 0.35
# ---------------------------

entrez_email <- email

# ---- Read gene list ----
genes_df <- read_csv(genes_csv, show_col_types = FALSE)
if (!"Gene" %in% names(genes_df)) stop('Input CSV must have a "Gene" column.')
genes <- genes_df$Gene %>% as.character() %>% trimws() %>% .[. != ""]

# ---- Query PubMed ----
results <- lapply(seq_along(genes), function(i) {
  gene <- genes[i]
  q <- sprintf(query_template, gene, condition)
  count <- NA_integer_
  status <- "Error"
  try({
    res <- entrez_search(db = "pubmed", term = q, retmax = 0)
    count <- as.integer(res$count)
    status <- ifelse(count > 0, "Yes", "No records found")
  }, silent = TRUE)
  if (i %% 50 == 0) message(sprintf("Processed %d / %d", i, length(genes)))
  Sys.sleep(sleep_sec)
  tibble(Gene = gene, Condition = condition, Query = q,
         PubMed_Count = ifelse(is.na(count), NA, count),
         Has_Papers = status)
})

# ---- Save results ----
bind_rows(results) %>% write_csv(out_csv)
cat("✅ Done! Results saved to:", out_csv, "\n")


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘XML’



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Processed 50 / 517

Processed 100 / 517

Processed 150 / 517

Processed 200 / 517

Processed 250 / 517

Processed 300 / 517

Processed 350 / 517

Processed 400 / 517

Processed 450 / 517

Processed 500 / 517



✅ Done! Results saved to: pubmed_gene_condition_hits.csv 


In [ ]:
# ---- Load libraries ----
library(rentrez)
library(readr)
library(dplyr)

# ---------- CONFIG ----------
email <- "your_email@example.com"     # 👈 Replace with your email (required by NCBI)
# Optional: Set NCBI API key to increase rate limits (get one from your NCBI account)
# Sys.setenv(NCBI_API_KEY = "your_ncbi_api_key_here")
Sys.setenv(NCBI_API_KEY = Sys.getenv("NCBI_API_KEY"))

condition <- "Psoriasis"              # 👈 Change to any condition you want
genes_csv <- "/content/genes.csv"              # CSV with column "Gene"
out_csv <- "pubmed_UR_hits.csv"
query_template <- '("%s"[All Fields]) AND ("%s"[All Fields])'
sleep_sec <- if (nzchar(Sys.getenv("NCBI_API_KEY"))) 0.12 else 0.35
# ---------------------------

entrez_email <- email

# ---- Read gene list ----
genes_df <- read_csv(genes_csv, show_col_types = FALSE)
if (!"Gene" %in% names(genes_df)) stop('Input CSV must have a "Gene" column.')
genes <- genes_df$Gene %>% as.character() %>% trimws() %>% .[. != ""]

# ---- Query PubMed ----
results <- lapply(seq_along(genes), function(i) {
  gene <- genes[i]
  q <- sprintf(query_template, gene, condition)
  count <- NA_integer_
  status <- "Error"
  try({
    res <- entrez_search(db = "pubmed", term = q, retmax = 0)
    count <- as.integer(res$count)
    status <- ifelse(count > 0, "Yes", "No records found")
  }, silent = TRUE)
  if (i %% 50 == 0) message(sprintf("Processed %d / %d", i, length(genes)))
  Sys.sleep(sleep_sec)
  tibble(Gene = gene, Condition = condition, Query = q,
         PubMed_Count = ifelse(is.na(count), NA, count),
         Has_Papers = status)
})

# ---- Save results ----
bind_rows(results) %>% write_csv(out_csv)
cat("✅ Done! Results saved to:", out_csv, "\n")


Processed 50 / 455

Processed 100 / 455

Processed 150 / 455

Processed 200 / 455

Processed 250 / 455

Processed 300 / 455

Processed 350 / 455

Processed 400 / 455

Processed 450 / 455



✅ Done! Results saved to: pubmed_UR_hits.csv 


# **PATHWAYS MAPPING**

In [ ]:
###############################################
# PubMed Pathway ↔ Psoriasis Mapper (R / Colab)
###############################################

# ---- Install (if needed) & load packages ----
need <- c("rentrez", "readr", "dplyr", "stringr", "tibble", "purrr")
inst <- need[!(need %in% installed.packages()[, "Package"])]
if (length(inst)) install.packages(inst, repos = "https://cloud.r-project.org")

library(rentrez)
library(readr)
library(dplyr)
library(stringr)
library(tibble)
library(purrr)

# ---------- CONFIG ----------
email <- "your_email@example.com"   # NCBI requires a contact email
# Optional: boost rate limits with your API key
# Sys.setenv(NCBI_API_KEY = "your_ncbi_api_key")
sleep_sec <- if (nzchar(Sys.getenv("NCBI_API_KEY"))) 0.12 else 0.35

condition <- "Psoriasis"            # change if needed
in_csv    <- "/content/Pathways.csv"         # CSV with column "Pathway"
out_csv   <- "pubmed_pathway_psoriasis_hits.csv"

# Query templates:
Q_TA   <- '("%s"[Title/Abstract]) AND ("%s"[Title/Abstract])'
Q_ALL  <- '("%s"[All Fields]) AND ("%s"[All Fields])'
# Optionally prefer MeSH for condition:
# Q_TA <- '("%s"[Title/Abstract]) AND ("Psoriasis"[Mesh] OR psoriasis[Title/Abstract] OR psoriatic[Title/Abstract])'

entrez_email <- email
# --------------------------------------------

# ---- Load pathways ----
df <- read_csv(in_csv, show_col_types = FALSE)
if (!"Pathway" %in% names(df)) stop('Input CSV must have a "Pathway" column.')

# Clean/normalize pathway strings a bit (keep as-is for exact phrase search)
clean <- function(x) {
  x %>%
    as.character() %>%
    str_squish() %>%
    str_replace_all("[\u2018\u2019]", "'") %>%  # curly quotes → straight
    str_replace_all('[\u201C\u201D]', '"')     # curly double-quotes → straight
}

pathways <- df$Pathway %>% clean() %>% .[. != ""]

# ---- Helper to get PubMed count ----
pm_count <- function(term) {
  out <- tryCatch({
    h <- entrez_search(db = "pubmed", term = term, retmax = 0)
    as.integer(h$count)
  }, error = function(e) NA_integer_)
  out
}

# ---- Search each pathway (two-stage: TA → All) ----
results <- imap_dfr(pathways, function(pw, i) {
  # Stage 1: Title/Abstract
  q_ta  <- sprintf(Q_TA, pw, condition)
  c_ta  <- pm_count(q_ta)

  # Stage 2: fallback to All Fields if TA returned 0 or NA
  need_fallback <- is.na(c_ta) || c_ta == 0
  q_all <- if (need_fallback) sprintf(Q_ALL, pw, condition) else NA_character_
  c_all <- if (need_fallback) pm_count(q_all) else NA_integer_

  Sys.sleep(sleep_sec)
  if (i %% 50 == 0) message(sprintf("Processed %d / %d", i, length(pathways)))

  # Final decision: known if TA>0 or ALL>0
  known <- ((is.integer(c_ta) && !is.na(c_ta) && c_ta > 0) ||
            (is.integer(c_all) && !is.na(c_all) && c_all > 0))

  tibble(
    Pathway           = pw,
    Condition         = condition,
    Query_TitleAbs    = q_ta,
    Count_TitleAbs    = c_ta,
    Query_AllFields   = q_all,
    Count_AllFields   = c_all,
    Known_in_PubMed   = ifelse(known, "Yes", "No records found")
  )
})

# ---- Save ----
write_csv(results, out_csv)
cat("✅ Done. Results saved to", out_csv, "\n")

# Tip: in Colab, copy to /content for easy download (optional)
system(paste("cp", shQuote(out_csv), "/content/"))


Processed 50 / 52



✅ Done. Results saved to pubmed_pathway_psoriasis_hits.csv 


# **IPA CP, DB AND ML Pathways Mapping**

In [ ]:
###############################################
# PubMed Pathway ↔ Psoriasis Mapper (R / Colab)
###############################################

# ---- Install (if needed) & load packages ----
need <- c("rentrez", "readr", "dplyr", "stringr", "tibble", "purrr")
inst <- need[!(need %in% installed.packages()[, "Package"])]
if (length(inst)) install.packages(inst, repos = "https://cloud.r-project.org")

library(rentrez)
library(readr)
library(dplyr)
library(stringr)
library(tibble)
library(purrr)

# ---------- CONFIG ----------
email <- "your_email@example.com"   # NCBI requires a contact email
# Optional: boost rate limits with your API key
# Sys.setenv(NCBI_API_KEY = "your_ncbi_api_key")
sleep_sec <- if (nzchar(Sys.getenv("NCBI_API_KEY"))) 0.12 else 0.35

condition <- "Psoriasis"            # change if needed
in_csv    <- "/content/Pathway.csv"         # CSV with column "Pathway"
out_csv   <- "pubmed_pathway_psoriasis_hits.csv"

# Query templates:
Q_TA   <- '("%s"[Title/Abstract]) AND ("%s"[Title/Abstract])'
Q_ALL  <- '("%s"[All Fields]) AND ("%s"[All Fields])'
# Optionally prefer MeSH for condition:
# Q_TA <- '("%s"[Title/Abstract]) AND ("Psoriasis"[Mesh] OR psoriasis[Title/Abstract] OR psoriatic[Title/Abstract])'

entrez_email <- email
# --------------------------------------------

# ---- Load pathways ----
df <- read_csv(in_csv, show_col_types = FALSE)
if (!"Pathway" %in% names(df)) stop('Input CSV must have a "Pathway" column.')

# Clean/normalize pathway strings a bit (keep as-is for exact phrase search)
clean <- function(x) {
  x %>%
    as.character() %>%
    str_squish() %>%
    str_replace_all("[\u2018\u2019]", "'") %>%  # curly quotes → straight
    str_replace_all('[\u201C\u201D]', '"')     # curly double-quotes → straight
}

pathways <- df$Pathway %>% clean() %>% .[. != ""]

# ---- Helper to get PubMed count ----
pm_count <- function(term) {
  out <- tryCatch({
    h <- entrez_search(db = "pubmed", term = term, retmax = 0)
    as.integer(h$count)
  }, error = function(e) NA_integer_)
  out
}

# ---- Search each pathway (two-stage: TA → All) ----
results <- imap_dfr(pathways, function(pw, i) {
  # Stage 1: Title/Abstract
  q_ta  <- sprintf(Q_TA, pw, condition)
  c_ta  <- pm_count(q_ta)

  # Stage 2: fallback to All Fields if TA returned 0 or NA
  need_fallback <- is.na(c_ta) || c_ta == 0
  q_all <- if (need_fallback) sprintf(Q_ALL, pw, condition) else NA_character_
  c_all <- if (need_fallback) pm_count(q_all) else NA_integer_

  Sys.sleep(sleep_sec)
  if (i %% 50 == 0) message(sprintf("Processed %d / %d", i, length(pathways)))

  # Final decision: known if TA>0 or ALL>0
  known <- ((is.integer(c_ta) && !is.na(c_ta) && c_ta > 0) ||
            (is.integer(c_all) && !is.na(c_all) && c_all > 0))

  tibble(
    Pathway           = pw,
    Condition         = condition,
    Query_TitleAbs    = q_ta,
    Count_TitleAbs    = c_ta,
    Query_AllFields   = q_all,
    Count_AllFields   = c_all,
    Known_in_PubMed   = ifelse(known, "Yes", "No records found")
  )
})

# ---- Save ----
write_csv(results, out_csv)
cat("✅ Done. Results saved to", out_csv, "\n")

# Tip: in Colab, copy to /content for easy download (optional)
system(paste("cp", shQuote(out_csv), "/content/"))


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘XML’



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Processed 50 / 73



✅ Done. Results saved to pubmed_pathway_psoriasis_hits.csv 


Wilms

In [ ]:
#!/usr/bin/env python3
"""
Search PubMed for papers relevant to:

    Inflammatory Gene Variants in Wilms Tumor - A Public Database Analysis

without any external input file.

It runs several thematic PubMed queries around:
- Wilms tumor / nephroblastoma
- Inflammation & tumor microenvironment
- COX-2 / PTGS2 / PTGER2 / PTGER4
- Macrophage-related genes (CCL2, CSF1R, ARG1)
- JAK-STAT signaling (STAT3, IL6, SOCS3)
- Immune checkpoints (PDCD1, CD274)
- Germline variants / polymorphisms / exome / SNP

Outputs (deduplicated PMIDs with basic metadata):
    WilmsTumor_inflammatory_variants_pubmed.csv
    WilmsTumor_inflammatory_variants_pubmed.xlsx
"""

import os
import time
import csv
import sys
import requests
import pandas as pd

# -------------------- CONFIG --------------------
NCBI_EMAIL = "raysona07@gmail.com"  # your email for NCBI
API_KEY = os.environ.get("NCBI_API_KEY", "").strip()

OUT_CSV  = "WilmsTumor_inflammatory_variants_pubmed.csv"
OUT_XLSX = "WilmsTumor_inflammatory_variants_pubmed.xlsx"

RETMAX_PER_QUERY = 300  # how many PMIDs per thematic query
SLEEP_PUBMED = 0.35 if API_KEY else 0.6  # respect NCBI rate limits
EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

# -------------------- THEMATIC QUERIES --------------------
def get_thematic_queries():
    """
    Define multiple themed PubMed queries based on the research question.
    Each entry: (label, pubmed_query_string)
    """

    wilms_block = (
        '(Wilms tumor[Title/Abstract] OR "Wilms tumor"[MeSH Terms] '
        'OR nephroblastoma[Title/Abstract] OR nephroblastoma[MeSH Terms])'
    )

    # General inflammation / tumor microenvironment
    general_inflamm = (
        f'{wilms_block} AND '
        '('
        'inflammat*[Title/Abstract] OR "tumor microenvironment"[Title/Abstract] '
        'OR macrophage*[Title/Abstract] OR "tumor-associated macrophage*"[Title/Abstract] '
        'OR "immune microenvironment"[Title/Abstract]'
        ')'
    )

    # COX-2 / PTGS2 / receptors
    cox2_block = (
        f'{wilms_block} AND '
        '('
        '"COX-2"[Title/Abstract] OR PTGS2[Title/Abstract] OR PTGER2[Title/Abstract] '
        'OR PTGER4[Title/Abstract]'
        ')'
    )

    # Macrophage-related genes
    macrophage_genes = (
        f'{wilms_block} AND '
        '('
        'CCL2[Title/Abstract] OR CSF1R[Title/Abstract] OR ARG1[Title/Abstract] '
        'OR "tumor-associated macrophage*"[Title/Abstract]'
        ')'
    )

    # JAK-STAT genes
    jak_stat_block = (
        f'{wilms_block} AND '
        '('
        'STAT3[Title/Abstract] OR IL6[Title/Abstract] OR SOCS3[Title/Abstract] '
        'OR "JAK-STAT"[Title/Abstract]'
        ')'
    )

    # Immune checkpoint molecules
    checkpoint_block = (
        f'{wilms_block} AND '
        '('
        'PDCD1[Title/Abstract] OR PD-1[Title/Abstract] '
        'OR CD274[Title/Abstract] OR PD-L1[Title/Abstract]'
        ')'
    )

    # Germline variants / polymorphisms / exome around Wilms tumor
    germline_variants = (
        f'{wilms_block} AND '
        '('
        'variant*[Title/Abstract] OR polymorphism*[Title/Abstract] '
        'OR germline[Title/Abstract] OR SNP[Title/Abstract] '
        'OR "single nucleotide polymorphism"[Title/Abstract] '
        'OR exome[Title/Abstract] OR "whole exome"[Title/Abstract]'
        ')'
    )

    return [
        ("General_inflammation_TME", general_inflamm),
        ("COX2_pathway", cox2_block),
        ("Macrophage_genes", macrophage_genes),
        ("JAK_STAT_signaling", jak_stat_block),
        ("Immune_checkpoints", checkpoint_block),
        ("Germline_variants", germline_variants),
    ]


# -------------------- BASIC HELPERS --------------------
def eutils_request(path, params, retries=3):
    """Wrapper for NCBI E-utilities with retries."""
    params = params.copy()
    params["email"] = NCBI_EMAIL
    if API_KEY:
        params["api_key"] = API_KEY
    url = f"{EUTILS}/{path}"
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            if attempt == retries:
                print(f"❌ NCBI request failed after {retries} attempts: {e}")
                raise
            time.sleep(SLEEP_PUBMED * attempt)


def esearch(query, retmax=50):
    """PubMed ESearch: return a list of PMIDs for a query."""
    params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "retmax": retmax,
        "sort": "relevance",
    }
    r = eutils_request("esearch.fcgi", params).json()
    return r.get("esearchresult", {}).get("idlist", [])


def esummary(pmids):
    """PubMed ESummary: title, journal, year for PMIDs."""
    if not pmids:
        return {}
    params = {"db": "pubmed", "id": ",".join(pmids), "retmode": "json"}
    r = eutils_request("esummary.fcgi", params).json()
    res = r.get("result", {})
    out = {}
    for pmid in pmids:
        d = res.get(pmid, {})
        title = d.get("title", "")
        journal = d.get("fulljournalname", d.get("source", ""))
        pubdate = d.get("pubdate", "")
        year = ""
        for token in str(pubdate).split():
            if token.isdigit() and len(token) == 4:
                year = token
                break
        out[pmid] = {"title": title, "journal": journal, "year": year}
    return out


def pubmed_url(pmid):
    return f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else ""


# -------------------- MAIN SEARCH LOGIC --------------------
def main():
    thematic_queries = get_thematic_queries()

    # Map PMID -> set of thematic labels that returned it
    pmid_to_themes = {}
    # Also track which themes had zero hits
    zero_hit_themes = []

    print("🔍 Running PubMed thematic searches for Wilms tumor inflammatory variants...\n")

    for label, query in thematic_queries:
        print(f"▶ Theme: {label}")
        print(f"   Query: {query}\n")
        try:
            pmids = esearch(query, retmax=RETMAX_PER_QUERY)
            time.sleep(SLEEP_PUBMED)
        except Exception as e:
            print(f"   ❌ Error in ESearch for theme {label}: {e}")
            zero_hit_themes.append((label, query, f"ESearch error: {e}"))
            continue

        if not pmids:
            print(f"   ⚠ No hits for theme: {label}\n")
            zero_hit_themes.append((label, query, "No hits"))
            continue

        print(f"   ✅ Retrieved {len(pmids)} PMIDs for theme: {label}\n")

        for pmid in pmids:
            if pmid not in pmid_to_themes:
                pmid_to_themes[pmid] = set()
            pmid_to_themes[pmid].add(label)

    all_pmids = list(pmid_to_themes.keys())
    print(f"\n📌 Total unique PMIDs across all themes: {len(all_pmids)}")

    # Fetch summary metadata for all unique PMIDs
    if all_pmids:
        # To be safe with very large lists, you could chunk this, but for a few hundred it's fine
        try:
            summaries = esummary(all_pmids)
            time.sleep(SLEEP_PUBMED)
        except Exception as e:
            print(f"❌ Error in ESummary: {e}")
            summaries = {}
    else:
        summaries = {}

    # -------------------- BUILD OUTPUT ROWS --------------------
    out_rows = []

    # Rows for real PMIDs
    for pmid, themes in pmid_to_themes.items():
        m = summaries.get(pmid, {})
        out_rows.append({
            "PMID": pmid,
            "Title": m.get("title", ""),
            "Journal": m.get("journal", ""),
            "Year": m.get("year", ""),
            "PubMed_URL": pubmed_url(pmid),
            "Themes": "; ".join(sorted(themes)),
            "In_Research_Scope": "YES",
            "Note": ""
        })

    # Optional: add rows for themes that had zero hits or errors
    for label, query, msg in zero_hit_themes:
        out_rows.append({
            "PMID": "NA",
            "Title": "NA",
            "Journal": "NA",
            "Year": "NA",
            "PubMed_URL": "NA",
            "Themes": label,
            "In_Research_Scope": "NO",
            "Note": msg + " | " + query
        })

    # -------------------- EXPORT --------------------
    cols = [
        "PMID", "Title", "Journal", "Year",
        "PubMed_URL", "Themes", "In_Research_Scope", "Note"
    ]

    df = pd.DataFrame(out_rows, columns=cols)
    df.replace('"', "'", regex=True, inplace=True)  # avoid weird quote issues

    # Excel
    df.to_excel(OUT_XLSX, index=False)

    # CSV with strong quoting
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, quoting=csv.QUOTE_ALL)
        w.writeheader()
        for row in out_rows:
            safe_row = {
                k: (str(v).replace('"', "'") if isinstance(v, str) else v)
                for k, v in row.items()
            }
            w.writerow(safe_row)

    print("\n✅ Completed!")
    print(f"Saved CSV : {OUT_CSV}")
    print(f"Saved XLSX: {OUT_XLSX}")


if __name__ == "__main__":
    main()


🔍 Running PubMed thematic searches for Wilms tumor inflammatory variants...

▶ Theme: General_inflammation_TME
   Query: (Wilms tumor[Title/Abstract] OR "Wilms tumor"[MeSH Terms] OR nephroblastoma[Title/Abstract] OR nephroblastoma[MeSH Terms]) AND (inflammat*[Title/Abstract] OR "tumor microenvironment"[Title/Abstract] OR macrophage*[Title/Abstract] OR "tumor-associated macrophage*"[Title/Abstract] OR "immune microenvironment"[Title/Abstract])

   ✅ Retrieved 300 PMIDs for theme: General_inflammation_TME

▶ Theme: COX2_pathway
   Query: (Wilms tumor[Title/Abstract] OR "Wilms tumor"[MeSH Terms] OR nephroblastoma[Title/Abstract] OR nephroblastoma[MeSH Terms]) AND ("COX-2"[Title/Abstract] OR PTGS2[Title/Abstract] OR PTGER2[Title/Abstract] OR PTGER4[Title/Abstract])

   ✅ Retrieved 12 PMIDs for theme: COX2_pathway

▶ Theme: Macrophage_genes
   Query: (Wilms tumor[Title/Abstract] OR "Wilms tumor"[MeSH Terms] OR nephroblastoma[Title/Abstract] OR nephroblastoma[MeSH Terms]) AND (CCL2[Title/Abs

In [ ]:
#!/usr/bin/env python3
"""
Search PubMed for papers relevant to:

    Longitudinal host transcriptomic and pathway changes
    in SIV/HIV infection and ART (acute infection, short-term ART, long-term ART)

This script:
- Runs several themed PubMed queries for:
  * HIV/SIV pathogenesis
  * ART, immune activation & residual inflammation
  * Neurological / neuroinflammatory complications
  * Cancer / oncogenic remodeling under HIV/ART
  * Transcriptomics (RNA-seq) & pathway analysis (IPA)
- Deduplicates PMIDs across themes
- Fetches metadata and constructs BibTeX entries

Outputs:
    SIV_ART_intro_pubmed.csv
    SIV_ART_intro_pubmed.xlsx

Columns:
    PMID, Title, Journal, Year, FirstAuthor, DOI, PubMed_URL,
    ArticleType, Themes, BibTeX_Key, BibTeX
"""

import os
import time
import csv
import sys
import requests
import pandas as pd
import xml.etree.ElementTree as ET

# -------------------- CONFIG --------------------
NCBI_EMAIL = "raysona07@gmail.com"  # your email for NCBI
API_KEY = os.environ.get("NCBI_API_KEY", "").strip()

OUT_CSV  = "SIV_ART_intro_pubmed.csv"
OUT_XLSX = "SIV_ART_intro_pubmed.xlsx"

RETMAX_PER_QUERY = 200  # how many PMIDs per thematic query
SLEEP_PUBMED = 0.35 if API_KEY else 0.6  # respect NCBI rate limits
EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

# -------------------- THEMATIC QUERIES --------------------
def get_thematic_queries():
    """
    Define multiple themed PubMed queries based on the SIV/HIV–ART introduction needs.
    Each entry: (label, pubmed_query_string)
    """

    hiv_siv_block = (
        '('
        'HIV[Title/Abstract] OR "Human immunodeficiency virus"[Title/Abstract] '
        'OR "HIV Infections"[MeSH Terms] OR "Acquired Immunodeficiency Syndrome"[MeSH Terms] '
        'OR SIV[Title/Abstract] OR "Simian immunodeficiency virus"[Title/Abstract]'
        ')'
    )

    macaque_block = (
        '('
        '"Macaca mulatta"[Title/Abstract] OR macaque*[Title/Abstract] '
        'OR "nonhuman primate*"[Title/Abstract]'
        ')'
    )

    # 1. General HIV/SIV pathogenesis, acute and chronic infection
    pathogenesis = (
        f'{hiv_siv_block} AND '
        '('
        'pathogenesis[Title/Abstract] OR "acute infection"[Title/Abstract] '
        'OR "chronic infection"[Title/Abstract] OR viremia[Title/Abstract] '
        'OR "immune activation"[Title/Abstract]'
        ')'
    )

    # 2. ART, immune activation, residual inflammation, long-term effects
    art_immune = (
        f'{hiv_siv_block} AND '
        '('
        '"antiretroviral therapy"[Title/Abstract] OR ART[Title/Abstract] '
        'OR cART[Title/Abstract] OR "combined antiretroviral"[Title/Abstract]'
        ') AND '
        '('
        '"immune activation"[Title/Abstract] OR inflammation[Title/Abstract] '
        'OR "residual inflammation"[Title/Abstract] OR "immune recovery"[Title/Abstract] '
        'OR "viral suppression"[Title/Abstract]'
        ')'
    )

    # 3. Neurocognitive / neuroinflammatory complications in HIV/SIV
    neuro_block = (
        f'{hiv_siv_block} AND '
        '('
        'neurocognitive[Title/Abstract] OR "neurocognitive disorder"[Title/Abstract] '
        'OR neuroinflammation[Title/Abstract] OR "HIV-associated neurocognitive"[Title/Abstract] '
        'OR "central nervous system"[Title/Abstract] OR CNS[Title/Abstract]'
        ')'
    )

    # 4. Cancer / oncogenic risk, cardiovascular & comorbidities under ART
    oncogenic_block = (
        f'{hiv_siv_block} AND '
        '('
        'cancer[Title/Abstract] OR oncogenic[Title/Abstract] OR neoplasm*[Title/Abstract] '
        'OR "non-AIDS-defining"[Title/Abstract] OR "cardiovascular disease"[Title/Abstract]'
        ') AND '
        '('
        '"antiretroviral therapy"[Title/Abstract] OR ART[Title/Abstract]'
        ')'
    )

    # 5. Transcriptomics / RNA-seq in HIV/SIV
    transcriptomics_block = (
        f'{hiv_siv_block} AND '
        '('
        '"RNA-seq"[Title/Abstract] OR "RNA sequencing"[Title/Abstract] '
        'OR transcriptome[Title/Abstract] OR transcriptomic*[Title/Abstract] '
        'OR "gene expression profiling"[Title/Abstract]'
        ')'
    )

    # 6. SIV macaque model + transcriptomics/pathways
    siv_macaque_transcriptomics = (
        f'{hiv_siv_block} AND {macaque_block} AND '
        '('
        '"RNA-seq"[Title/Abstract] OR transcriptomic*[Title/Abstract] '
        'OR microarray[Title/Abstract] OR "gene expression"[Title/Abstract]'
        ')'
    )

    # 7. IPA / pathway analysis in HIV/SIV
    ipa_block = (
        f'{hiv_siv_block} AND '
        '('
        '"Ingenuity Pathway Analysis"[Title/Abstract] OR IPA[Title/Abstract] '
        'OR "pathway analysis"[Title/Abstract] OR "canonical pathway"[Title/Abstract]'
        ')'
    )

    return [
        ("Pathogenesis_acute_chronic", pathogenesis),
        ("ART_immune_activation", art_immune),
        ("Neurocognitive_neuroinflammation", neuro_block),
        ("Oncogenic_comorbidities_ART", oncogenic_block),
        ("HIV_SIV_transcriptomics", transcriptomics_block),
        ("SIV_macaque_transcriptomics", siv_macaque_transcriptomics),
        ("IPA_pathway_analysis", ipa_block),
    ]

# -------------------- BASIC HELPERS --------------------
def eutils_request(path, params, retries=3):
    """Wrapper for NCBI E-utilities with retries."""
    params = params.copy()
    params["email"] = NCBI_EMAIL
    if API_KEY:
        params["api_key"] = API_KEY
    url = f"{EUTILS}/{path}"
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            if attempt == retries:
                print(f"❌ NCBI request failed after {retries} attempts: {e}")
                raise
            time.sleep(SLEEP_PUBMED * attempt)


def esearch(query, retmax=50):
    """PubMed ESearch: return a list of PMIDs for a query."""
    params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "retmax": retmax,
        "sort": "relevance",
    }
    r = eutils_request("esearch.fcgi", params).json()
    return r.get("esearchresult", {}).get("idlist", [])


def pubmed_url(pmid):
    return f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else ""


# -------------------- EFetch + BibTeX CONSTRUCTION --------------------
def efetch_details(pmids):
    """
    Use PubMed EFetch (XML) to get more detailed metadata and build BibTeX-like entries.
    Returns dict[pmid] = {
        "Title", "Journal", "Year", "FirstAuthor", "DOI",
        "ArticleType", "BibTeX_Key", "BibTeX"
    }
    """
    if not pmids:
        return {}

    out = {}
    # Chunk PMIDs in case there are many
    CHUNK_SIZE = 200
    for i in range(0, len(pmids), CHUNK_SIZE):
        chunk = pmids[i:i + CHUNK_SIZE]
        params = {
            "db": "pubmed",
            "id": ",".join(chunk),
            "retmode": "xml",
        }
        r = eutils_request("efetch.fcgi", params)
        root = ET.fromstring(r.text)

        for article in root.findall(".//PubmedArticle"):
            try:
                pmid_elem = article.find(".//PMID")
                pmid = pmid_elem.text.strip() if pmid_elem is not None else None
                if not pmid:
                    continue

                art = article.find(".//Article")
                if art is None:
                    continue

                # Title
                title_elem = art.find(".//ArticleTitle")
                title = "".join(title_elem.itertext()).strip() if title_elem is not None else ""

                # Journal
                journal_elem = art.find(".//Journal")
                journal_title_elem = journal_elem.find("Title") if journal_elem is not None else None
                journal = journal_title_elem.text.strip() if journal_title_elem is not None else ""

                # Year
                pub_date = journal_elem.find("JournalIssue/PubDate") if journal_elem is not None else None
                year = ""
                if pub_date is not None:
                    y = pub_date.find("Year")
                    if y is not None and y.text and y.text.strip().isdigit():
                        year = y.text.strip()
                    else:
                        # Sometimes only MedlineDate is given, like "1998 Jan-Feb"
                        md = pub_date.find("MedlineDate")
                        if md is not None and md.text:
                            for tok in md.text.split():
                                if tok.isdigit() and len(tok) == 4:
                                    year = tok
                                    break

                # Pages, volume, issue
                medline_pg = art.find(".//Pagination/MedlinePgn")
                pages = medline_pg.text.strip() if medline_pg is not None else ""
                vol = art.find(".//JournalIssue/Volume")
                volume = vol.text.strip() if vol is not None else ""
                iss = art.find(".//JournalIssue/Issue")
                issue = iss.text.strip() if iss is not None else ""

                # Authors
                author_list = art.findall(".//AuthorList/Author")
                authors = []
                first_author_lastname = ""
                for idx, au in enumerate(author_list):
                    last = au.find("LastName")
                    fore = au.find("ForeName")
                    initials = au.find("Initials")

                    if last is None:
                        continue
                    last_name = last.text.strip()
                    if fore is not None and fore.text:
                        first_name = fore.text.strip()
                    elif initials is not None and initials.text:
                        first_name = initials.text.strip()
                    else:
                        first_name = ""
                    if idx == 0:
                        first_author_lastname = last_name
                    if first_name:
                        authors.append(f"{last_name}, {first_name}")
                    else:
                        authors.append(last_name)

                authors_str = " and ".join(authors)

                # DOI
                doi = ""
                for aid in article.findall(".//ArticleIdList/ArticleId"):
                    if aid.get("IdType") == "doi" and aid.text:
                        doi = aid.text.strip()
                        break

                # Article type (Review vs Original)
                pub_types = [
                    pt.text.strip() for pt in article.findall(".//PublicationTypeList/PublicationType")
                    if pt is not None and pt.text
                ]
                if any("Review" in pt for pt in pub_types):
                    article_type = "Review"
                else:
                    article_type = "Original"

                # BibTeX key
                # e.g. FirstAuthorYYYY_JournalAbbrevWithoutSpaces
                # Fallback if missing year or author
                key_author = first_author_lastname if first_author_lastname else "UnknownAuthor"
                key_year = year if year else "n.d."
                journal_abbrev = journal.replace(" ", "")[:20]
                bib_key = f"{key_author}{key_year}_{journal_abbrev}"
                # Sanitize key
                bib_key = "".join(ch for ch in bib_key if ch.isalnum() or ch in "_.-")

                # Escape braces in text fields
                def esc(s):
                    return s.replace("{", "").replace("}", "")

                # Construct BibTeX
                bib_lines = [
                    f"@article{{{bib_key},",
                    f"  author  = {{{esc(authors_str)}}},",
                    f"  title   = {{{esc(title)}}},",
                    f"  journal = {{{esc(journal)}}},",
                    f"  year    = {{{year}}},",
                    f"  volume  = {{{volume}}},",
                    f"  number  = {{{issue}}},",
                    f"  pages   = {{{pages}}},",
                    f"  doi     = {{{doi}}},",
                    f"  pmid    = {{{pmid}}}",
                    "}"
                ]
                bibtex_str = "\n".join(bib_lines)

                out[pmid] = {
                    "Title": title,
                    "Journal": journal,
                    "Year": year,
                    "FirstAuthor": first_author_lastname,
                    "DOI": doi,
                    "ArticleType": article_type,
                    "BibTeX_Key": bib_key,
                    "BibTeX": bibtex_str,
                }

            except Exception as e:
                # Be robust and continue even if one record fails
                print(f"⚠ Error parsing article in EFetch: {e}")
                continue

        time.sleep(SLEEP_PUBMED)

    return out

# -------------------- MAIN SEARCH LOGIC --------------------
def main():
    thematic_queries = get_thematic_queries()

    # Map PMID -> set of thematic labels that returned it
    pmid_to_themes = {}
    zero_hit_themes = []

    print("🔍 Running PubMed thematic searches for SIV/HIV–ART introduction...\n")

    for label, query in thematic_queries:
        print(f"▶ Theme: {label}")
        print(f"   Query: {query}\n")
        try:
            pmids = esearch(query, retmax=RETMAX_PER_QUERY)
            time.sleep(SLEEP_PUBMED)
        except Exception as e:
            print(f"   ❌ Error in ESearch for theme {label}: {e}")
            zero_hit_themes.append((label, query, f"ESearch error: {e}"))
            continue

        if not pmids:
            print(f"   ⚠ No hits for theme: {label}\n")
            zero_hit_themes.append((label, query, "No hits"))
            continue

        print(f"   ✅ Retrieved {len(pmids)} PMIDs for theme: {label}\n")

        for pmid in pmids:
            if pmid not in pmid_to_themes:
                pmid_to_themes[pmid] = set()
            pmid_to_themes[pmid].add(label)

    all_pmids = list(pmid_to_themes.keys())
    print(f"\n📌 Total unique PMIDs across all themes: {len(all_pmids)}")

    # Fetch detailed metadata + BibTeX
    if all_pmids:
        try:
            details = efetch_details(all_pmids)
        except Exception as e:
            print(f"❌ Error in EFetch/BibTeX step: {e}")
            details = {}
    else:
        details = {}

    # -------------------- BUILD OUTPUT ROWS --------------------
    out_rows = []

    # Rows for real PMIDs
    for pmid, themes in pmid_to_themes.items():
        m = details.get(pmid, {})
        out_rows.append({
            "PMID": pmid,
            "Title": m.get("Title", ""),
            "Journal": m.get("Journal", ""),
            "Year": m.get("Year", ""),
            "FirstAuthor": m.get("FirstAuthor", ""),
            "DOI": m.get("DOI", ""),
            "PubMed_URL": pubmed_url(pmid),
            "ArticleType": m.get("ArticleType", ""),
            "Themes": "; ".join(sorted(themes)),
            "BibTeX_Key": m.get("BibTeX_Key", ""),
            "BibTeX": m.get("BibTeX", ""),
        })

    # Optional: add rows for themes that had zero hits or errors
    for label, query, msg in zero_hit_themes:
        out_rows.append({
            "PMID": "NA",
            "Title": "NA",
            "Journal": "NA",
            "Year": "NA",
            "FirstAuthor": "NA",
            "DOI": "NA",
            "PubMed_URL": "NA",
            "ArticleType": "NA",
            "Themes": label,
            "BibTeX_Key": "NA",
            "BibTeX": msg + " | " + query,
        })

    # -------------------- EXPORT --------------------
    cols = [
        "PMID", "Title", "Journal", "Year", "FirstAuthor",
        "DOI", "PubMed_URL", "ArticleType",
        "Themes", "BibTeX_Key", "BibTeX"
    ]

    df = pd.DataFrame(out_rows, columns=cols)
    df.replace('"', "'", regex=True, inplace=True)  # avoid quote issues

    # Excel
    df.to_excel(OUT_XLSX, index=False)

    # CSV with strong quoting
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, quoting=csv.QUOTE_ALL)
        w.writeheader()
        for row in out_rows:
            safe_row = {
                k: (str(v).replace('"', "'") if isinstance(v, str) else v)
                for k, v in row.items()
            }
            w.writerow(safe_row)

    print("\n✅ Completed!")
    print(f"Saved CSV : {OUT_CSV}")
    print(f"Saved XLSX: {OUT_XLSX}")


if __name__ == "__main__":
    main()


🔍 Running PubMed thematic searches for SIV/HIV–ART introduction...

▶ Theme: Pathogenesis_acute_chronic
   Query: (HIV[Title/Abstract] OR "Human immunodeficiency virus"[Title/Abstract] OR "HIV Infections"[MeSH Terms] OR "Acquired Immunodeficiency Syndrome"[MeSH Terms] OR SIV[Title/Abstract] OR "Simian immunodeficiency virus"[Title/Abstract]) AND (pathogenesis[Title/Abstract] OR "acute infection"[Title/Abstract] OR "chronic infection"[Title/Abstract] OR viremia[Title/Abstract] OR "immune activation"[Title/Abstract])

   ✅ Retrieved 200 PMIDs for theme: Pathogenesis_acute_chronic

▶ Theme: ART_immune_activation
   Query: (HIV[Title/Abstract] OR "Human immunodeficiency virus"[Title/Abstract] OR "HIV Infections"[MeSH Terms] OR "Acquired Immunodeficiency Syndrome"[MeSH Terms] OR SIV[Title/Abstract] OR "Simian immunodeficiency virus"[Title/Abstract]) AND ("antiretroviral therapy"[Title/Abstract] OR ART[Title/Abstract] OR cART[Title/Abstract] OR "combined antiretroviral"[Title/Abstract]) AND 

In [ ]:
#!/usr/bin/env python3
"""
Search PubMed for papers relevant to:

    Longitudinal host transcriptomic and pathway changes
    in SIV/HIV infection and ART (acute infection, short-term ART, long-term ART)

This script:
- Runs several themed PubMed queries for:
  * HIV/SIV pathogenesis
  * ART, immune activation & residual inflammation
  * Neurological / neuroinflammatory complications
  * Cancer / oncogenic remodeling under HIV/ART
  * Transcriptomics (RNA-seq) & pathway analysis (IPA)
- Deduplicates PMIDs across themes
- Fetches metadata and constructs BibTeX entries

Outputs:
    SIV_ART_intro_pubmed.csv
    SIV_ART_intro_pubmed.xlsx

Columns:
    PMID, Title, Journal, Year, FirstAuthor, DOI, PubMed_URL,
    ArticleType, Themes, BibTeX_Key, BibTeX
"""

import os
import time
import csv
import sys
import requests
import pandas as pd
import xml.etree.ElementTree as ET

# -------------------- CONFIG --------------------
NCBI_EMAIL = "raysona07@gmail.com"  # your email for NCBI
API_KEY = os.environ.get("NCBI_API_KEY", "").strip()

OUT_CSV  = "SIV_ART_intro_pubmed.csv"
OUT_XLSX = "SIV_ART_intro_pubmed.xlsx"

RETMAX_PER_QUERY = 200  # how many PMIDs per thematic query
SLEEP_PUBMED = 0.35 if API_KEY else 0.6  # respect NCBI rate limits
EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

# -------------------- THEMATIC QUERIES --------------------
def get_thematic_queries():
    """
    Define multiple themed PubMed queries based on the SIV/HIV–ART introduction needs.
    Each entry: (label, pubmed_query_string)
    """

    hiv_siv_block = (
        '('
        'HIV[Title/Abstract] OR "Human immunodeficiency virus"[Title/Abstract] '
        'OR "HIV Infections"[MeSH Terms] OR "Acquired Immunodeficiency Syndrome"[MeSH Terms] '
        'OR SIV[Title/Abstract] OR "Simian immunodeficiency virus"[Title/Abstract]'
        ')'
    )

    macaque_block = (
        '('
        '"Macaca mulatta"[Title/Abstract] OR macaque*[Title/Abstract] '
        'OR "nonhuman primate*"[Title/Abstract]'
        ')'
    )

    # 1. General HIV/SIV pathogenesis, acute and chronic infection
    pathogenesis = (
        f'{hiv_siv_block} AND '
        '('
        'pathogenesis[Title/Abstract] OR "acute infection"[Title/Abstract] '
        'OR "chronic infection"[Title/Abstract] OR viremia[Title/Abstract] '
        'OR "immune activation"[Title/Abstract]'
        ')'
    )

    # 2. ART, immune activation, residual inflammation, long-term effects
    art_immune = (
        f'{hiv_siv_block} AND '
        '('
        '"antiretroviral therapy"[Title/Abstract] OR ART[Title/Abstract] '
        'OR cART[Title/Abstract] OR "combined antiretroviral"[Title/Abstract]'
        ') AND '
        '('
        '"immune activation"[Title/Abstract] OR inflammation[Title/Abstract] '
        'OR "residual inflammation"[Title/Abstract] OR "immune recovery"[Title/Abstract] '
        'OR "viral suppression"[Title/Abstract]'
        ')'
    )

    # 3. Neurocognitive / neuroinflammatory complications in HIV/SIV
    neuro_block = (
        f'{hiv_siv_block} AND '
        '('
        'neurocognitive[Title/Abstract] OR "neurocognitive disorder"[Title/Abstract] '
        'OR neuroinflammation[Title/Abstract] OR "HIV-associated neurocognitive"[Title/Abstract] '
        'OR "central nervous system"[Title/Abstract] OR CNS[Title/Abstract]'
        ')'
    )

    # 4. Cancer / oncogenic risk, cardiovascular & comorbidities under ART
    oncogenic_block = (
        f'{hiv_siv_block} AND '
        '('
        'cancer[Title/Abstract] OR oncogenic[Title/Abstract] OR neoplasm*[Title/Abstract] '
        'OR "non-AIDS-defining"[Title/Abstract] OR "cardiovascular disease"[Title/Abstract]'
        ') AND '
        '('
        '"antiretroviral therapy"[Title/Abstract] OR ART[Title/Abstract]'
        ')'
    )

    # 5. Transcriptomics / RNA-seq in HIV/SIV
    transcriptomics_block = (
        f'{hiv_siv_block} AND '
        '('
        '"RNA-seq"[Title/Abstract] OR "RNA sequencing"[Title/Abstract] '
        'OR transcriptome[Title/Abstract] OR transcriptomic*[Title/Abstract] '
        'OR "gene expression profiling"[Title/Abstract]'
        ')'
    )

    # 6. SIV macaque model + transcriptomics/pathways
    siv_macaque_transcriptomics = (
        f'{hiv_siv_block} AND {macaque_block} AND '
        '('
        '"RNA-seq"[Title/Abstract] OR transcriptomic*[Title/Abstract] '
        'OR microarray[Title/Abstract] OR "gene expression"[Title/Abstract]'
        ')'
    )

    # 7. IPA / pathway analysis in HIV/SIV
    ipa_block = (
        f'{hiv_siv_block} AND '
        '('
        '"Ingenuity Pathway Analysis"[Title/Abstract] OR IPA[Title/Abstract] '
        'OR "pathway analysis"[Title/Abstract] OR "canonical pathway"[Title/Abstract]'
        ')'
    )

    return [
        ("Pathogenesis_acute_chronic", pathogenesis),
        ("ART_immune_activation", art_immune),
        ("Neurocognitive_neuroinflammation", neuro_block),
        ("Oncogenic_comorbidities_ART", oncogenic_block),
        ("HIV_SIV_transcriptomics", transcriptomics_block),
        ("SIV_macaque_transcriptomics", siv_macaque_transcriptomics),
        ("IPA_pathway_analysis", ipa_block),
    ]

# -------------------- BASIC HELPERS --------------------
def eutils_request(path, params, retries=3):
    """Wrapper for NCBI E-utilities with retries."""
    params = params.copy()
    params["email"] = NCBI_EMAIL
    if API_KEY:
        params["api_key"] = API_KEY
    url = f"{EUTILS}/{path}"
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            if attempt == retries:
                print(f"❌ NCBI request failed after {retries} attempts: {e}")
                raise
            time.sleep(SLEEP_PUBMED * attempt)


def esearch(query, retmax=50):
    """PubMed ESearch: return a list of PMIDs for a query."""
    params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "retmax": retmax,
        "sort": "relevance",
    }
    r = eutils_request("esearch.fcgi", params).json()
    return r.get("esearchresult", {}).get("idlist", [])


def pubmed_url(pmid):
    return f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else ""


# -------------------- EFetch + BibTeX CONSTRUCTION --------------------
def efetch_details(pmids):
    """
    Use PubMed EFetch (XML) to get more detailed metadata and build BibTeX-like entries.
    Returns dict[pmid] = {
        "Title", "Journal", "Year", "FirstAuthor", "DOI",
        "ArticleType", "BibTeX_Key", "BibTeX"
    }
    """
    if not pmids:
        return {}

    out = {}
    # Chunk PMIDs in case there are many
    CHUNK_SIZE = 200
    for i in range(0, len(pmids), CHUNK_SIZE):
        chunk = pmids[i:i + CHUNK_SIZE]
        params = {
            "db": "pubmed",
            "id": ",".join(chunk),
            "retmode": "xml",
        }
        r = eutils_request("efetch.fcgi", params)
        root = ET.fromstring(r.text)

        for article in root.findall(".//PubmedArticle"):
            try:
                pmid_elem = article.find(".//PMID")
                pmid = pmid_elem.text.strip() if pmid_elem is not None else None
                if not pmid:
                    continue

                art = article.find(".//Article")
                if art is None:
                    continue

                # Title
                title_elem = art.find(".//ArticleTitle")
                title = "".join(title_elem.itertext()).strip() if title_elem is not None else ""

                # Journal
                journal_elem = art.find(".//Journal")
                journal_title_elem = journal_elem.find("Title") if journal_elem is not None else None
                journal = journal_title_elem.text.strip() if journal_title_elem is not None else ""

                # Year
                pub_date = journal_elem.find("JournalIssue/PubDate") if journal_elem is not None else None
                year = ""
                if pub_date is not None:
                    y = pub_date.find("Year")
                    if y is not None and y.text and y.text.strip().isdigit():
                        year = y.text.strip()
                    else:
                        # Sometimes only MedlineDate is given, like "1998 Jan-Feb"
                        md = pub_date.find("MedlineDate")
                        if md is not None and md.text:
                            for tok in md.text.split():
                                if tok.isdigit() and len(tok) == 4:
                                    year = tok
                                    break

                # Pages, volume, issue
                medline_pg = art.find(".//Pagination/MedlinePgn")
                pages = medline_pg.text.strip() if medline_pg is not None else ""
                vol = art.find(".//JournalIssue/Volume")
                volume = vol.text.strip() if vol is not None else ""
                iss = art.find(".//JournalIssue/Issue")
                issue = iss.text.strip() if iss is not None else ""

                # Authors
                author_list = art.findall(".//AuthorList/Author")
                authors = []
                first_author_lastname = ""
                for idx, au in enumerate(author_list):
                    last = au.find("LastName")
                    fore = au.find("ForeName")
                    initials = au.find("Initials")

                    if last is None:
                        continue
                    last_name = last.text.strip()
                    if fore is not None and fore.text:
                        first_name = fore.text.strip()
                    elif initials is not None and initials.text:
                        first_name = initials.text.strip()
                    else:
                        first_name = ""
                    if idx == 0:
                        first_author_lastname = last_name
                    if first_name:
                        authors.append(f"{last_name}, {first_name}")
                    else:
                        authors.append(last_name)

                authors_str = " and ".join(authors)

                # DOI
                doi = ""
                for aid in article.findall(".//ArticleIdList/ArticleId"):
                    if aid.get("IdType") == "doi" and aid.text:
                        doi = aid.text.strip()
                        break

                # Article type (Review vs Original)
                pub_types = [
                    pt.text.strip() for pt in article.findall(".//PublicationTypeList/PublicationType")
                    if pt is not None and pt.text
                ]
                if any("Review" in pt for pt in pub_types):
                    article_type = "Review"
                else:
                    article_type = "Original"

                # BibTeX key
                # e.g. FirstAuthorYYYY_JournalAbbrevWithoutSpaces
                # Fallback if missing year or author
                key_author = first_author_lastname if first_author_lastname else "UnknownAuthor"
                key_year = year if year else "n.d."
                journal_abbrev = journal.replace(" ", "")[:20]
                bib_key = f"{key_author}{key_year}_{journal_abbrev}"
                # Sanitize key
                bib_key = "".join(ch for ch in bib_key if ch.isalnum() or ch in "_.-")

                # Escape braces in text fields
                def esc(s):
                    return s.replace("{", "").replace("}", "")

                # Construct BibTeX
                bib_lines = [
                    f"@article{{{bib_key},",
                    f"  author  = {{{esc(authors_str)}}},",
                    f"  title   = {{{esc(title)}}},",
                    f"  journal = {{{esc(journal)}}},",
                    f"  year    = {{{year}}},",
                    f"  volume  = {{{volume}}},",
                    f"  number  = {{{issue}}},",
                    f"  pages   = {{{pages}}},",
                    f"  doi     = {{{doi}}},",
                    f"  pmid    = {{{pmid}}}",
                    "}"
                ]
                bibtex_str = "\n".join(bib_lines)

                out[pmid] = {
                    "Title": title,
                    "Journal": journal,
                    "Year": year,
                    "FirstAuthor": first_author_lastname,
                    "DOI": doi,
                    "ArticleType": article_type,
                    "BibTeX_Key": bib_key,
                    "BibTeX": bibtex_str,
                }

            except Exception as e:
                # Be robust and continue even if one record fails
                print(f"⚠ Error parsing article in EFetch: {e}")
                continue

        time.sleep(SLEEP_PUBMED)

    return out

# -------------------- MAIN SEARCH LOGIC --------------------
def main():
    thematic_queries = get_thematic_queries()

    # Map PMID -> set of thematic labels that returned it
    pmid_to_themes = {}
    zero_hit_themes = []

    print("🔍 Running PubMed thematic searches for SIV/HIV–ART introduction...\n")

    for label, query in thematic_queries:
        print(f"▶ Theme: {label}")
        print(f"   Query: {query}\n")
        try:
            pmids = esearch(query, retmax=RETMAX_PER_QUERY)
            time.sleep(SLEEP_PUBMED)
        except Exception as e:
            print(f"   ❌ Error in ESearch for theme {label}: {e}")
            zero_hit_themes.append((label, query, f"ESearch error: {e}"))
            continue

        if not pmids:
            print(f"   ⚠ No hits for theme: {label}\n")
            zero_hit_themes.append((label, query, "No hits"))
            continue

        print(f"   ✅ Retrieved {len(pmids)} PMIDs for theme: {label}\n")

        for pmid in pmids:
            if pmid not in pmid_to_themes:
                pmid_to_themes[pmid] = set()
            pmid_to_themes[pmid].add(label)

    all_pmids = list(pmid_to_themes.keys())
    print(f"\n📌 Total unique PMIDs across all themes: {len(all_pmids)}")

    # Fetch detailed metadata + BibTeX
    if all_pmids:
        try:
            details = efetch_details(all_pmids)
        except Exception as e:
            print(f"❌ Error in EFetch/BibTeX step: {e}")
            details = {}
    else:
        details = {}

    # -------------------- BUILD OUTPUT ROWS --------------------
    out_rows = []

    # Rows for real PMIDs
    for pmid, themes in pmid_to_themes.items():
        m = details.get(pmid, {})
        out_rows.append({
            "PMID": pmid,
            "Title": m.get("Title", ""),
            "Journal": m.get("Journal", ""),
            "Year": m.get("Year", ""),
            "FirstAuthor": m.get("FirstAuthor", ""),
            "DOI": m.get("DOI", ""),
            "PubMed_URL": pubmed_url(pmid),
            "ArticleType": m.get("ArticleType", ""),
            "Themes": "; ".join(sorted(themes)),
            "BibTeX_Key": m.get("BibTeX_Key", ""),
            "BibTeX": m.get("BibTeX", ""),
        })

    # Optional: add rows for themes that had zero hits or errors
    for label, query, msg in zero_hit_themes:
        out_rows.append({
            "PMID": "NA",
            "Title": "NA",
            "Journal": "NA",
            "Year": "NA",
            "FirstAuthor": "NA",
            "DOI": "NA",
            "PubMed_URL": "NA",
            "ArticleType": "NA",
            "Themes": label,
            "BibTeX_Key": "NA",
            "BibTeX": msg + " | " + query,
        })

    # -------------------- EXPORT --------------------
    cols = [
        "PMID", "Title", "Journal", "Year", "FirstAuthor",
        "DOI", "PubMed_URL", "ArticleType",
        "Themes", "BibTeX_Key", "BibTeX"
    ]

    df = pd.DataFrame(out_rows, columns=cols)
    df.replace('"', "'", regex=True, inplace=True)  # avoid quote issues

    # Excel
    df.to_excel(OUT_XLSX, index=False)

    # CSV with strong quoting
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, quoting=csv.QUOTE_ALL)
        w.writeheader()
        for row in out_rows:
            safe_row = {
                k: (str(v).replace('"', "'") if isinstance(v, str) else v)
                for k, v in row.items()
            }
            w.writerow(safe_row)

    print("\n✅ Completed!")
    print(f"Saved CSV : {OUT_CSV}")
    print(f"Saved XLSX: {OUT_XLSX}")


if __name__ == "__main__":
    main()


🔍 Running PubMed thematic searches for SIV/HIV–ART introduction...

▶ Theme: Pathogenesis_acute_chronic
   Query: (HIV[Title/Abstract] OR "Human immunodeficiency virus"[Title/Abstract] OR "HIV Infections"[MeSH Terms] OR "Acquired Immunodeficiency Syndrome"[MeSH Terms] OR SIV[Title/Abstract] OR "Simian immunodeficiency virus"[Title/Abstract]) AND (pathogenesis[Title/Abstract] OR "acute infection"[Title/Abstract] OR "chronic infection"[Title/Abstract] OR viremia[Title/Abstract] OR "immune activation"[Title/Abstract])

   ✅ Retrieved 200 PMIDs for theme: Pathogenesis_acute_chronic

▶ Theme: ART_immune_activation
   Query: (HIV[Title/Abstract] OR "Human immunodeficiency virus"[Title/Abstract] OR "HIV Infections"[MeSH Terms] OR "Acquired Immunodeficiency Syndrome"[MeSH Terms] OR SIV[Title/Abstract] OR "Simian immunodeficiency virus"[Title/Abstract]) AND ("antiretroviral therapy"[Title/Abstract] OR ART[Title/Abstract] OR cART[Title/Abstract] OR "combined antiretroviral"[Title/Abstract]) AND 